In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from tqdm import tqdm

In [ ]:
train = pd.read_csv('/content/train.csv')
# year, month, item_id 기준으로 value 합산 (seq만 다르다면 value 합산)
monthly = (
    train
    .groupby(["item_id", "year", "month"], as_index=False)["value"]
    .sum()
)

# year, month를 하나의 키(ym)로 묶기
monthly["ym"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2)
)

# item_id × ym 피벗 (월별 총 무역량 매트릭스 생성)
pivot = (
    monthly
    .pivot(index="item_id", columns="ym", values="value")
    .fillna(0.0)
)

pivot.head()

ym,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,...,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01
item_id,,,,,,,,,,,,,,,,,,,,,
AANGBULD,14276.0,52347.0,53549.0,0.0,26997.0,84489.0,0.0,0.0,0.0,0.0,...,428725.0,144248.0,26507.0,25691.0,25805.0,0.0,38441.0,0.0,441275.0,533478.0
AHMDUILJ,242705.0,120847.0,197317.0,126142.0,71730.0,149138.0,186617.0,169995.0,140547.0,89292.0,...,123085.0,143451.0,78649.0,125098.0,80404.0,157401.0,115509.0,127473.0,89479.0,101317.0
ANWUJOKX,0.0,0.0,0.0,63580.0,81670.0,26424.0,8470.0,0.0,0.0,80475.0,...,0.0,0.0,0.0,27980.0,0.0,0.0,0.0,0.0,0.0,0.0
APQGTRMF,383999.0,512813.0,217064.0,470398.0,539873.0,582317.0,759980.0,216019.0,537693.0,205326.0,...,683581.0,2147.0,0.0,25013.0,77.0,20741.0,2403.0,3543.0,32430.0,40608.0
ATLDMDBO,143097177.0,103568323.0,118403737.0,121873741.0,115024617.0,65716075.0,146216818.0,97552978.0,72341427.0,87454167.0,...,60276050.0,30160198.0,42613728.0,64451013.0,38667429.0,29354408.0,42450439.0,37136720.0,32181798.0,57090235.0


In [ ]:
# 이걸로 해봄  - 0.3794
from scipy.stats import spearmanr

def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


def safe_corr_lag(x, y, lag, max_zero_ratio=0.5, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    L = len(x_shifted)
    if L < min_valid:
        return 0.0

    if np.std(x_shifted) == 0 or np.std(y_shifted) == 0:
        return 0.0

    zero_ratio = np.mean((x_shifted == 0) & (y_shifted == 0))
    if zero_ratio > max_zero_ratio:
        return 0.0

    # Pearson
    pear = float(np.corrcoef(x_shifted, y_shifted)[0, 1])

    # Spearman
    spear = spearmanr(x_shifted, y_shifted).correlation
    if spear is None or np.isnan(spear):
        return 0.0

    # 🔥 부호가 다르면 "진짜 공행성"이 아님 → 버림
    if pear * spear <= 0:
        return 0.0

    # Pearson 값만 반환 (threshold는 pear 기준)
    return pear




def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_valid_lag=9
):
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0

            for lag in range(1, max_lag + 1):
                if n_months <= lag:
                    continue

                corr = safe_corr_lag(x, y, lag)

                # lag 구간별 threshold 적용
                th = dynamic_threshold(lag)

                if abs(corr) >= th:
                    # best correlation 업데이트
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag is not None:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    return pd.DataFrame(results)



# 실행
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()


100it [01:02,  1.60it/s]

탐색된 공행성쌍 수: 2765


,leading_item_id,following_item_id,best_lag,max_corr
0,AANGBULD,APQGTRMF,8,-0.480115
1,AANGBULD,BEZYMBBT,10,-0.483240
2,AANGBULD,BLANHGYY,11,0.588076
3,AANGBULD,DDEXPPXU,2,0.383169
4,AANGBULD,DEWLVASR,6,0.640221


In [ ]:
#3392행인데 0.3717 점 코
def dynamic_threshold(lag):
    """lag 구간별 threshold"""
    if lag <= 6:
        return 0.35
    else:  # 7~12
        return 0.35


def safe_corr_lag(x, y, lag, max_zero_ratio=0.5, min_valid=9):
    """
    lag 적용 시 앞뒤 구간 제거 → 나머지 구간에서 Pearson 상관계수 계산
    """
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    L = len(x_shifted)
    if L < min_valid:
        return 0.0

    if np.std(x_shifted) == 0 or np.std(y_shifted) == 0:
        return 0.0

    zero_ratio = np.mean((x_shifted == 0) & (y_shifted == 0))
    if zero_ratio > max_zero_ratio:
        return 0.0

    return float(np.corrcoef(x_shifted, y_shifted)[0, 1])



def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_valid_lag=9
):
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0

            for lag in range(1, max_lag + 1):
                if n_months <= lag:
                    continue

                corr = safe_corr_lag(x, y, lag)

                # lag 구간별 threshold 적용
                th = dynamic_threshold(lag)

                if abs(corr) >= th:
                    # best correlation 업데이트
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag is not None:
                results.append({
                    "leading_item_id": leader,
                    "following_item_id": follower,
                    "best_lag": best_lag,
                    "max_corr": best_corr,
                })

    return pd.DataFrame(results)



# 실행
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()


100it [00:15,  6.59it/s]

탐색된 공행성쌍 수: 3392


,leading_item_id,following_item_id,best_lag,max_corr
0,AANGBULD,APQGTRMF,8,-0.480115
1,AANGBULD,BEZYMBBT,10,-0.483240
2,AANGBULD,BJALXPFS,12,0.397642
3,AANGBULD,BLANHGYY,11,0.588076
4,AANGBULD,DBWLZWNK,9,-0.366877


In [ ]:
# 이걸로 해보기 롤링기법으로 패턴이 일정한지를 12개월을 기준으로 확인하느 코드
# 이게 점수 2등 0.382
# + 수정 해서 한번 해보기? sign 비율 0.6으로 바꿔서
#######################################
from scipy.stats import spearmanr

# 1. lag 구간별 threshold
#######################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#######################################
# 2. main 상관계수 (Pearson + Spearman sign check)
#######################################
def safe_corr_lag(x, y, lag, max_zero_ratio=0.5, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    L = len(x_shifted)
    if L < min_valid:
        return 0.0

    if np.std(x_shifted) == 0 or np.std(y_shifted) == 0:
        return 0.0

    zero_ratio = np.mean((x_shifted == 0) & (y_shifted == 0))
    if zero_ratio > max_zero_ratio:
        return 0.0

    ## Pearson
    pear = float(np.corrcoef(x_shifted, y_shifted)[0, 1])

    ## Spearman
    spear = spearmanr(x_shifted, y_shifted).correlation
    if spear is None or np.isnan(spear):
        return 0.0

    ## 🔥 부호 다르면 진짜 공행성 아님 → 제거
    if pear * spear <= 0:
        return 0.0

    return pear  # Pearson만 반환


#######################################
# 3. Rolling Sign Consistency 계산
#######################################
def rolling_sign_consistency(x, y, lag, window=12, min_windows=5):
    """
    best_lag에서 rolling corr을 구해 sign 안정성을 계산.
    """
    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    T = len(x_shifted)
    if T <= window:
        return 0.0

    roll_corrs = []
    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        c = np.corrcoef(xs, ys)[0, 1]
        if np.isnan(c):
            continue

        roll_corrs.append(c)

    if len(roll_corrs) < min_windows:
        return 0.0

    return roll_corrs


#######################################
# 4. 공행성쌍 탐색
#######################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_valid_lag=9,
    min_consistency=0.65   # 🔥 sign 안정성 기준
):
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0

            ## best lag 찾기
            for lag in range(1, max_lag + 1):
                if lag >= n_months:
                    continue

                corr = safe_corr_lag(x, y, lag)

                th = dynamic_threshold(lag)

                if abs(corr) >= th:
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag is None:
                continue

            ###########################################
            # 🔥 추가: rolling sign consistency 필터
            ###########################################
            roll_corrs = rolling_sign_consistency(x, y, best_lag)

            if len(roll_corrs) == 0:
                continue

            sign_main = np.sign(best_corr)
            same_sign_ratio = np.mean(np.sign(roll_corrs) == sign_main)

            if same_sign_ratio < min_consistency:
                continue

            ###########################################

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "max_corr": best_corr,
                "sign_consistency": same_sign_ratio
            })

    return pd.DataFrame(results)



#######################################
# 실행
#######################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()


100it [01:14,  1.34it/s]

탐색된 공행성쌍 수: 2320


,leading_item_id,following_item_id,best_lag,max_corr,sign_consistency
0,AANGBULD,APQGTRMF,8,-0.480115,0.958333
1,AANGBULD,BLANHGYY,11,0.588076,0.809524
2,AANGBULD,DDEXPPXU,2,0.383169,0.900000
3,AANGBULD,DEWLVASR,6,0.640221,0.807692
4,AANGBULD,DNMPSKTB,4,-0.410635,0.964286


In [ ]:
#treshold 를 다 0.35를 기준으
#점수 0.376
#######################################
from scipy.stats import spearmanr

# 1. lag 구간별 threshold
#######################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.35


#######################################
# 2. main 상관계수 (Pearson + Spearman sign check)
#######################################
def safe_corr_lag(x, y, lag, max_zero_ratio=0.5, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    L = len(x_shifted)
    if L < min_valid:
        return 0.0

    if np.std(x_shifted) == 0 or np.std(y_shifted) == 0:
        return 0.0

    zero_ratio = np.mean((x_shifted == 0) & (y_shifted == 0))
    if zero_ratio > max_zero_ratio:
        return 0.0

    ## Pearson
    pear = float(np.corrcoef(x_shifted, y_shifted)[0, 1])

    ## Spearman
    spear = spearmanr(x_shifted, y_shifted).correlation
    if spear is None or np.isnan(spear):
        return 0.0

    ## 🔥 부호 다르면 진짜 공행성 아님 → 제거
    if pear * spear <= 0:
        return 0.0

    return pear  # Pearson만 반환


#######################################
# 3. Rolling Sign Consistency 계산
#######################################
def rolling_sign_consistency(x, y, lag, window=12, min_windows=5):
    """
    best_lag에서 rolling corr을 구해 sign 안정성을 계산.
    """
    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    T = len(x_shifted)
    if T <= window:
        return 0.0

    roll_corrs = []
    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        c = np.corrcoef(xs, ys)[0, 1]
        if np.isnan(c):
            continue

        roll_corrs.append(c)

    if len(roll_corrs) < min_windows:
        return 0.0

    return roll_corrs


#######################################
# 4. 공행성쌍 탐색
#######################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_valid_lag=9,
    min_consistency=0.65   # 🔥 sign 안정성 기준
):
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0

            ## best lag 찾기
            for lag in range(1, max_lag + 1):
                if lag >= n_months:
                    continue

                corr = safe_corr_lag(x, y, lag)

                th = dynamic_threshold(lag)

                if abs(corr) >= th:
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag is None:
                continue

            ###########################################
            # 🔥 추가: rolling sign consistency 필터
            ###########################################
            roll_corrs = rolling_sign_consistency(x, y, best_lag)

            if len(roll_corrs) == 0:
                continue

            sign_main = np.sign(best_corr)
            same_sign_ratio = np.mean(np.sign(roll_corrs) == sign_main)

            if same_sign_ratio < min_consistency:
                continue

            ###########################################

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "max_corr": best_corr,
                "sign_consistency": same_sign_ratio
            })

    return pd.DataFrame(results)



#######################################
# 실행
#######################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()


100it [01:19,  1.26it/s]

탐색된 공행성쌍 수: 2784


,leading_item_id,following_item_id,best_lag,max_corr,sign_consistency
0,AANGBULD,APQGTRMF,8,-0.480115,0.958333
1,AANGBULD,BLANHGYY,11,0.588076,0.809524
2,AANGBULD,DDEXPPXU,2,0.383169,0.900000
3,AANGBULD,DEWLVASR,6,0.640221,0.807692
4,AANGBULD,DNMPSKTB,4,-0.410635,0.964286


In [ ]:
#최고점수에서 lag를 18까지 확장
#점수 0.356
#######################################
from scipy.stats import spearmanr

# 1. lag 구간별 threshold
#######################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    elif lag <= 12:
        return 0.40
    else:
        return 0.5


#######################################
# 2. main 상관계수 (Pearson + Spearman sign check)
#######################################
def safe_corr_lag(x, y, lag, max_zero_ratio=0.5, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    L = len(x_shifted)
    if L < min_valid:
        return 0.0

    if np.std(x_shifted) == 0 or np.std(y_shifted) == 0:
        return 0.0

    zero_ratio = np.mean((x_shifted == 0) & (y_shifted == 0))
    if zero_ratio > max_zero_ratio:
        return 0.0

    ## Pearson
    pear = float(np.corrcoef(x_shifted, y_shifted)[0, 1])

    ## Spearman
    spear = spearmanr(x_shifted, y_shifted).correlation
    if spear is None or np.isnan(spear):
        return 0.0

    ## 🔥 부호 다르면 진짜 공행성 아님 → 제거
    if pear * spear <= 0:
        return 0.0

    return pear  # Pearson만 반환


#######################################
# 3. Rolling Sign Consistency 계산
#######################################
def rolling_sign_consistency(x, y, lag, window=12, min_windows=5):
    """
    best_lag에서 rolling corr을 구해 sign 안정성을 계산.
    """
    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    T = len(x_shifted)
    if T <= window:
        return 0.0

    roll_corrs = []
    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        c = np.corrcoef(xs, ys)[0, 1]
        if np.isnan(c):
            continue

        roll_corrs.append(c)

    if len(roll_corrs) < min_windows:
        return 0.0

    return roll_corrs


#######################################
# 4. 공행성쌍 탐색
#######################################
def find_comovement_pairs(
    pivot,
    max_lag=18,
    min_nonzero=12,
    max_zero=15,
    min_valid_lag=9,
    min_consistency=0.65   # 🔥 sign 안정성 기준
):
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0

            ## best lag 찾기
            for lag in range(1, max_lag + 1):
                if lag >= n_months:
                    continue

                corr = safe_corr_lag(x, y, lag)

                th = dynamic_threshold(lag)

                if abs(corr) >= th:
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag is None:
                continue

            ###########################################
            # 🔥 추가: rolling sign consistency 필터
            ###########################################
            roll_corrs = rolling_sign_consistency(x, y, best_lag)

            if len(roll_corrs) == 0:
                continue

            sign_main = np.sign(best_corr)
            same_sign_ratio = np.mean(np.sign(roll_corrs) == sign_main)

            if same_sign_ratio < min_consistency:
                continue

            ###########################################

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "max_corr": best_corr,
                "sign_consistency": same_sign_ratio
            })

    return pd.DataFrame(results)



#######################################
# 실행
#######################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()


100it [01:49,  1.10s/it]

탐색된 공행성쌍 수: 2616


,leading_item_id,following_item_id,best_lag,max_corr,sign_consistency
0,AANGBULD,APQGTRMF,8,-0.480115,0.958333
1,AANGBULD,DDEXPPXU,2,0.383169,0.900000
2,AANGBULD,DEWLVASR,6,0.640221,0.807692
3,AANGBULD,DNMPSKTB,4,-0.410635,0.964286
4,AANGBULD,ELQGMQWE,14,0.558635,0.833333


In [ ]:
# 점수 0.354
#######################################
# 1. lag 구간별 threshold
#######################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.3
    else:
        return 0.35


#######################################
# 2. main 상관계수 (Pearson + Spearman sign check)
#######################################
def safe_corr_lag(x, y, lag, max_zero_ratio=0.5, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    L = len(x_shifted)
    if L < min_valid:
        return 0.0

    if np.std(x_shifted) == 0 or np.std(y_shifted) == 0:
        return 0.0

    zero_ratio = np.mean((x_shifted == 0) & (y_shifted == 0))
    if zero_ratio > max_zero_ratio:
        return 0.0

    ## Pearson
    pear = float(np.corrcoef(x_shifted, y_shifted)[0, 1])

    ## Spearman
    spear = spearmanr(x_shifted, y_shifted).correlation
    if spear is None or np.isnan(spear):
        return 0.0

    ## 🔥 부호 다르면 진짜 공행성 아님 → 제거
    if pear * spear <= 0:
        return 0.0

    return pear  # Pearson만 반환


#######################################
# 3. Rolling Sign Consistency 계산
#######################################
def rolling_sign_consistency(x, y, lag, window=12, min_windows=5):
    """
    best_lag에서 rolling corr을 구해 sign 안정성을 계산.
    """
    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    T = len(x_shifted)
    if T <= window:
        return 0.0

    roll_corrs = []
    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        c = np.corrcoef(xs, ys)[0, 1]
        if np.isnan(c):
            continue

        roll_corrs.append(c)

    if len(roll_corrs) < min_windows:
        return 0.0

    return roll_corrs


#######################################
# 4. 공행성쌍 탐색
#######################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_valid_lag=9,
    min_consistency=0.65   # 🔥 sign 안정성 기준
):
    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_corr = 0.0

            ## best lag 찾기
            for lag in range(1, max_lag + 1):
                if lag >= n_months:
                    continue

                corr = safe_corr_lag(x, y, lag)

                th = dynamic_threshold(lag)

                if abs(corr) >= th:
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag is None:
                continue

            ###########################################
            # 🔥 추가: rolling sign consistency 필터
            ###########################################
            roll_corrs = rolling_sign_consistency(x, y, best_lag)

            if len(roll_corrs) == 0:
                continue

            sign_main = np.sign(best_corr)
            same_sign_ratio = np.mean(np.sign(roll_corrs) == sign_main)

            if same_sign_ratio < min_consistency:
                continue

            ###########################################

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "max_corr": best_corr,
                "sign_consistency": same_sign_ratio
            })

    return pd.DataFrame(results)



#######################################
# 실행
#######################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()


100it [01:23,  1.19it/s]

탐색된 공행성쌍 수: 3307


,leading_item_id,following_item_id,best_lag,max_corr,sign_consistency
0,AANGBULD,APQGTRMF,8,-0.480115,0.958333
1,AANGBULD,BLANHGYY,11,0.588076,0.809524
2,AANGBULD,DDEXPPXU,2,0.383169,0.900000
3,AANGBULD,DEWLVASR,6,0.640221,0.807692
4,AANGBULD,DNMPSKTB,4,-0.410635,0.964286


이거 제출하기 11/17일에

In [ ]:
# 이게 진짜 최고 코드 0.3837
# 부호를 고려하는 각 롤링창에서도 pearson x spearman >0 인 비율 까지 고려 이거는 해볼지 말지 나중에 생각하기
from scipy.stats import spearmanr

#######################################
# 1. lag 구간별 threshold
#######################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#######################################
# 2. Main Pearson + Spearman sign check
#######################################
def safe_corr_lag(x, y, lag, max_zero_ratio=0.5, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    L = len(x_shifted)

    if L < min_valid:
        return 0.0

    if np.std(x_shifted) == 0 or np.std(y_shifted) == 0:
        return 0.0

    zero_ratio = np.mean((x_shifted == 0) & (y_shifted == 0))
    if zero_ratio > max_zero_ratio:
        return 0.0

    pear = float(np.corrcoef(x_shifted, y_shifted)[0, 1])
    spear = spearmanr(x_shifted, y_shifted).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    # 💥 메인 부호 불일치 → 제거
    if pear * spear <= 0:
        return 0.0

    return pear


#######################################
# 3. Rolling 안정성 + Soft sign check
#######################################
def rolling_corr_metrics(x, y, lag, window=12, min_windows=5):
    """
    rolling windows:
      - pearson corr
      - pearson*spear_sign_ratio
    """
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)

    if T <= window:
        return None

    roll_corrs = []
    roll_sign_agree = []  # pearson*spear > 0 여부 저장

    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0, 1]
        spear = spearmanr(xs, ys).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        roll_corrs.append(pear)
        roll_sign_agree.append(pear * spear > 0)   # True/False 저장

    if len(roll_corrs) < min_windows:
        return None

    roll_corrs = np.array(roll_corrs)
    roll_sign_agree = np.array(roll_sign_agree)

    return {
        "corrs": roll_corrs,
        "sign_agree_ratio": roll_sign_agree.mean(),
        "std": roll_corrs.std(),
        "mean_abs": np.abs(roll_corrs).mean()
    }


#######################################
# 4. 공행성 탐색
#######################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,     # rolling pearson sign consistency
    min_sign_agree=0.60       # 🔥 rolling Pearson×Spearman의 soft sign agreement
):

    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            ###################################
            # best lag 찾기
            ###################################
            best_lag = None
            best_corr = 0.0

            for lag in range(1, max_lag + 1):
                if lag >= n_months:
                    continue

                corr = safe_corr_lag(x, y, lag)
                th = dynamic_threshold(lag)

                if abs(corr) >= th:
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag is None:
                continue

            ###################################
            # rolling 안정성 + soft sign 검사
            ###################################
            metrics = rolling_corr_metrics(x, y, best_lag)
            if metrics is None:
                continue

            sign_main = np.sign(best_corr)

            # pearson 기준 동일 부호 비율
            same_sign_ratio = np.mean(np.sign(metrics["corrs"]) == sign_main)

            if same_sign_ratio < min_consistency:
                continue

            # pearson*spear 부호 일치 비율 → soft check
            if metrics["sign_agree_ratio"] < min_sign_agree:
                continue

            ###################################
            # 최종 채택
            ###################################
            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "max_corr": best_corr,
                "roll_pearson_sign_ratio": same_sign_ratio,
                "roll_spearman_sign_ratio": metrics["sign_agree_ratio"],
                "roll_std": metrics["std"],
                "roll_mean_abs": metrics["mean_abs"]
            })

    return pd.DataFrame(results)


#######################################
# 실행
#######################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()


100it [01:48,  1.08s/it]

탐색된 공행성쌍 수: 2434


,leading_item_id,following_item_id,best_lag,max_corr,roll_pearson_sign_ratio,roll_spearman_sign_ratio,roll_std,roll_mean_abs
0,AANGBULD,APQGTRMF,8,-0.480115,0.958333,0.833333,0.183385,0.250263
1,AANGBULD,BLANHGYY,11,0.588076,0.809524,0.761905,0.264457,0.316035
2,AANGBULD,DDEXPPXU,2,0.383169,0.900000,0.833333,0.271188,0.387041
3,AANGBULD,DEWLVASR,6,0.640221,0.807692,0.884615,0.340005,0.496775
4,AANGBULD,DNMPSKTB,4,-0.410635,0.964286,0.857143,0.256069,0.378640


In [ ]:
#최고 코드에서 log를 취한걸로
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm

#######################################
# 1. lag 구간별 threshold
#######################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40

#######################################
# 2. Main Pearson + Spearman sign check
#######################################
def safe_corr_lag(x, y, lag, max_zero_ratio=0.5, min_valid=9):
    # x, y는 이미 log1p가 적용된 상태로 들어옴
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    L = len(x_shifted)

    if L < min_valid:
        return 0.0

    if np.std(x_shifted) == 0 or np.std(y_shifted) == 0:
        return 0.0

    # 0 비율 체크 (로그 변환 후에도 0은 0임)
    zero_ratio = np.mean((x_shifted == 0) & (y_shifted == 0))
    if zero_ratio > max_zero_ratio:
        return 0.0

    pear = float(np.corrcoef(x_shifted, y_shifted)[0, 1])
    spear = spearmanr(x_shifted, y_shifted).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    # 💥 메인 부호 불일치 → 제거
    if pear * spear <= 0:
        return 0.0

    return pear

#######################################
# 3. Rolling 안정성 + Soft sign check
#######################################
def rolling_corr_metrics(x, y, lag, window=12, min_windows=5):
    """
    rolling windows:
      - pearson corr (log applied)
      - pearson*spear_sign_ratio
    """
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)

    if T <= window:
        return None

    roll_corrs = []
    roll_sign_agree = []  # pearson*spear > 0 여부 저장

    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0, 1]
        spear = spearmanr(xs, ys).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        roll_corrs.append(pear)
        roll_sign_agree.append(pear * spear > 0)   # True/False 저장

    if len(roll_corrs) < min_windows:
        return None

    roll_corrs = np.array(roll_corrs)
    roll_sign_agree = np.array(roll_sign_agree)

    return {
        "corrs": roll_corrs,
        "sign_agree_ratio": roll_sign_agree.mean(),
        "std": roll_corrs.std(),
        "mean_abs": np.abs(roll_corrs).mean()
    }

#######################################
# 4. 공행성 탐색 (Log 적용 버전)
#######################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,     # rolling pearson sign consistency
    min_sign_agree=0.60       # 🔥 rolling Pearson×Spearman의 soft sign agreement
):

    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for i, leader in tqdm(enumerate(items)):
        # 1. Raw Data 추출 (0 개수 세기 용도)
        x_raw = pivot.loc[leader].values.astype(float)

        # 2. 0 개수 필터링 (Raw 데이터 기준)
        if np.sum(x_raw == 0) >= max_zero:
            continue
        if np.count_nonzero(x_raw) < min_nonzero:
            continue

        # 3. 🔥 로그 변환 적용 (np.log1p = log(x+1))
        x = np.log1p(x_raw)

        for follower in items:
            if follower == leader:
                continue

            # 1. Raw Data 추출
            y_raw = pivot.loc[follower].values.astype(float)

            # 2. 0 개수 필터링
            if np.sum(y_raw == 0) >= max_zero:
                continue
            if np.count_nonzero(y_raw) < min_nonzero:
                continue

            # 3. 🔥 로그 변환 적용
            y = np.log1p(y_raw)

            ###################################
            # best lag 찾기 (로그 데이터 x, y 사용)
            ###################################
            best_lag = None
            best_corr = 0.0

            for lag in range(1, max_lag + 1):
                if lag >= n_months:
                    continue

                # safe_corr_lag 안에서 log된 x, y로 상관계수 계산
                corr = safe_corr_lag(x, y, lag)
                th = dynamic_threshold(lag)

                if abs(corr) >= th:
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag is None:
                continue

            ###################################
            # rolling 안정성 + soft sign 검사
            ###################################
            metrics = rolling_corr_metrics(x, y, best_lag)
            if metrics is None:
                continue

            sign_main = np.sign(best_corr)

            # pearson 기준 동일 부호 비율
            same_sign_ratio = np.mean(np.sign(metrics["corrs"]) == sign_main)

            if same_sign_ratio < min_consistency:
                continue

            # pearson*spear 부호 일치 비율 → soft check
            if metrics["sign_agree_ratio"] < min_sign_agree:
                continue

            ###################################
            # 최종 채택
            ###################################
            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "max_corr": best_corr,
                "roll_pearson_sign_ratio": same_sign_ratio,
                "roll_spearman_sign_ratio": metrics["sign_agree_ratio"],
                "roll_std": metrics["std"],
                "roll_mean_abs": metrics["mean_abs"]
            })

    return pd.DataFrame(results)

#######################################
# 실행
#######################################
# pivot 데이터는 원본(raw) 값을 가지고 있어야 합니다.
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()

100it [02:04,  1.24s/it]

탐색된 공행성쌍 수: 2548


,leading_item_id,following_item_id,best_lag,max_corr,roll_pearson_sign_ratio,roll_spearman_sign_ratio,roll_std,roll_mean_abs
0,AANGBULD,APQGTRMF,5,-0.444719,0.962963,1.000000,0.118904,0.328386
1,AANGBULD,AXULOHBQ,1,-0.494126,0.935484,0.838710,0.201705,0.326067
2,AANGBULD,BLANHGYY,2,-0.384603,1.000000,1.000000,0.122668,0.371317
3,AANGBULD,BSRMSVTC,4,-0.415788,0.964286,0.892857,0.207828,0.348221
4,AANGBULD,DEWLVASR,6,0.378967,0.653846,0.961538,0.294591,0.284321


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm

EPS = 1e-6   # 로그 안정화용

################################################################################
# 1. Threshold
################################################################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


################################################################################
# 2. Level correlation (log 안정적)
################################################################################
def safe_corr_lag_level(x, y, lag, min_valid=9):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    xv = lx[:-lag]
    yv = ly[lag:]

    if len(xv) < min_valid:
        return 0.0
    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation
    if spear is None or np.isnan(spear):
        return 0.0

    if pear * spear <= 0:
        return 0.0

    return pear


################################################################################
# 3. Lag-diff correlation  (핵심!!  변화량(t)-변화량(t-lag))
################################################################################
def safe_corr_lag_diff(x, y, lag, min_valid=9):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    # lag-length 변화량
    dx = lx[lag:] - lx[:-lag]      # A[t] - A[t-lag]
    dy = ly[lag:] - ly[:-lag]      # B[t] - B[t-lag]

    # shift: A influences B after lag
    if len(dx) <= 2 * lag:
        return 0.0

    dx2 = dx[:-lag]
    dy2 = dy[lag:]

    if len(dx2) < min_valid:
        return 0.0

    mask = ~((dx2 == 0) & (dy2 == 0))
    dxv = dx2[mask]
    dyv = dy2[mask]

    if len(dxv) < min_valid:
        return 0.0
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    pear = float(np.corrcoef(dxv, dyv)[0, 1])
    spear = spearmanr(dxv, dyv).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    if pear * spear <= 0:
        return 0.0

    return pear


################################################################################
# 4. lag-diff sign ratio (변화 방향 동의율)
################################################################################
def lagdiff_sign_ratio(x, y, lag, min_valid=6):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = lx[lag:] - lx[:-lag]
    dy = ly[lag:] - ly[:-lag]

    if len(dx) <= 2 * lag:
        return 0.0

    dx2 = dx[:-lag]
    dy2 = dy[lag:]

    mask = ~((dx2 == 0) & (dy2 == 0))
    dxv = dx2[mask]
    dyv = dy2[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))


################################################################################
# 5-1 Rolling for Level
################################################################################
def rolling_level_metrics(x, y, lag, window=12, min_windows=5):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    xv = lx[:-lag]
    yv = ly[lag:]
    T = len(xv)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = xv[start:start+window]
        ys = yv[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0, 1]
        spear = spearmanr(xs, ys).correlation
        if np.isnan(pear) or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pa = np.array(pear_list)
    main_sign = np.sign(np.mean(pa))

    return {
        "pear_sign_ratio": np.mean(np.sign(pa) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


################################################################################
# 5-2 Rolling for Lag-Diff
################################################################################
def rolling_lagdiff_metrics(x, y, lag, window=12, min_windows=5):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = lx[lag:] - lx[:-lag]
    dy = ly[lag:] - ly[:-lag]

    dx2 = dx[:-lag]
    dy2 = dy[lag:]
    T = len(dx2)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = dx2[start:start+window]
        ys = dy2[start:start+window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5 or np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0, 1]
        spear = spearmanr(xs2, ys2).correlation
        if np.isnan(pear) or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pa = np.array(pear_list)
    main_sign = np.sign(np.mean(pa))

    return {
        "pear_sign_ratio": np.mean(np.sign(pa) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


################################################################################
# 6. MASTER SEARCH (Level + LagDiff 둘 모두)
################################################################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_pear_consistency=0.60,
    min_spear_agree=0.60,
    diff_sign_threshold=0.60
):

    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if leader == follower:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            for lag in range(1, max_lag + 1):

                th = dynamic_threshold(lag)

                ############################################################################
                # Level correlation
                ############################################################################
                lv_corr = safe_corr_lag_level(x, y, lag)
                s_lv = abs(lv_corr) if abs(lv_corr) >= th else 0

                ############################################################################
                # Lag-Diff correlation
                ############################################################################
                df_corr = safe_corr_lag_diff(x, y, lag)
                df_sign = lagdiff_sign_ratio(x, y, lag)
                s_df = abs(df_corr) if (abs(df_corr) >= th and df_sign >= diff_sign_threshold) else 0

                # Choose winner
                if s_df > s_lv:
                    score = s_df * np.sign(df_corr)
                    tp = "lagdiff"
                else:
                    score = s_lv * np.sign(lv_corr)
                    tp = "level"

                if abs(score) > abs(best_score):
                    best_score = score
                    best_lag = lag
                    best_type = tp

            if best_lag is None:
                continue

            ############################################################################
            # Rolling verification
            ############################################################################
            if best_type == "level":
                roll = rolling_level_metrics(x, y, best_lag)
            else:
                roll = rolling_lagdiff_metrics(x, y, best_lag)

            if roll is None:
                continue

            if roll["pear_sign_ratio"] < min_pear_consistency:
                continue
            if roll["spearman_agree_ratio"] < min_spear_agree:
                continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_pear_ratio": roll["pear_sign_ratio"],
                "roll_spear_ratio": roll["spearman_agree_ratio"]
            })

    return pd.DataFrame(results)


################################################################################
# RUN
################################################################################
pairs = find_comovement_pairs(pivot)
print("검출된 공행성 쌍:", len(pairs))
pairs.head()


100%|██████████| 100/100 [03:45<00:00,  2.26s/it]

검출된 공행성 쌍: 3285


,leading_item_id,following_item_id,best_lag,score,type,roll_pear_ratio,roll_spear_ratio
0,AANGBULD,AXULOHBQ,1,-0.459176,level,0.903226,0.806452
1,AANGBULD,BLANHGYY,5,0.507517,lagdiff,1.000000,1.000000
2,AANGBULD,BSRMSVTC,4,-0.417119,level,0.964286,0.892857
3,AANGBULD,DNMPSKTB,5,-0.350557,level,0.703704,0.851852
4,AANGBULD,ELQGMQWE,7,0.448132,level,1.000000,0.960000


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm

EPS = 1e-6


###############################################################################
# 1. Threshold
###############################################################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


###############################################################################
# 2. Level correlation (log 안정)
###############################################################################
def corr_level(x, y, lag, min_valid=9):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    xv = lx[:-lag]
    yv = ly[lag:]

    if len(xv) < min_valid:
        return None
    if np.std(xv) == 0 or np.std(yv) == 0:
        return None

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return None
    if pear * spear <= 0:
        return None

    return pear


###############################################################################
# 3. Lag-diff correlation (진짜 lag 기반 변화)
###############################################################################
def corr_lagdiff(x, y, lag, min_valid=9):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = lx[lag:] - lx[:-lag]
    dy = ly[lag:] - ly[:-lag]

    if len(dx) <= 2 * lag:
        return None

    dx2 = dx[:-lag]
    dy2 = dy[lag:]

    if len(dx2) < min_valid:
        return None

    mask = ~((dx2 == 0) & (dy2 == 0))
    dxv = dx2[mask]
    dyv = dy2[mask]

    if len(dxv) < min_valid:
        return None
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return None

    pear = float(np.corrcoef(dxv, dyv)[0, 1])
    spear = spearmanr(dxv, dyv).correlation
    if spear is None or np.isnan(spear):
        return None
    if pear * spear <= 0:
        return None

    return pear


###############################################################################
# 4. Rolling stability (level)
###############################################################################
def rolling_level(x, y, lag, window=12, min_windows=5):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    xv = lx[:-lag]
    yv = ly[lag:]
    T = len(xv)

    if T <= window:
        return None

    plist = []
    agree = []

    for s in range(0, T - window + 1):
        xs = xv[s:s+window]
        ys = yv[s:s+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0, 1]
        spear = spearmanr(xs, ys).correlation
        if np.isnan(pear) or np.isnan(spear):
            continue

        plist.append(pear)
        agree.append(pear * spear > 0)

    if len(plist) < min_windows:
        return None

    pa = np.array(plist)
    main_sign = np.sign(np.mean(pa))

    return {
        "pear_ratio": np.mean(np.sign(pa) == main_sign),
        "agree_ratio": np.mean(agree)
    }


###############################################################################
# 5. Rolling stability (lag-diff)
###############################################################################
def rolling_diff(x, y, lag, window=12, min_windows=5):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = lx[lag:] - lx[:-lag]
    dy = ly[lag:] - ly[:-lag]

    dx2 = dx[:-lag]
    dy2 = dy[lag:]
    T = len(dx2)

    if T <= window:
        return None

    plist = []
    agree = []

    for s in range(0, T - window + 1):
        xs = dx2[s:s+window]
        ys = dy2[s:s+window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0, 1]
        spear = spearmanr(xs2, ys2).correlation
        if np.isnan(pear) or np.isnan(spear):
            continue

        plist.append(pear)
        agree.append(pear * spear > 0)

    if len(plist) < min_windows:
        return None

    pa = np.array(plist)
    main_sign = np.sign(np.mean(pa))

    return {
        "pear_ratio": np.mean(np.sign(pa) == main_sign),
        "agree_ratio": np.mean(agree)
    }


###############################################################################
# 6. MASTER: Level + LagDiff 동시에 검사
###############################################################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_valid_lags=2,
    min_nonzero=12,
    max_zero=15,
    min_pear_stable=0.60,
    min_spear_stable=0.60
):

    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero or np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if leader == follower:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.sum(y == 0) >= max_zero or np.count_nonzero(y) < min_nonzero:
                continue

            level_good = []
            diff_good = []

            # ---------------------------
            # 모든 lag(1~12) 검사
            # ---------------------------
            for lag in range(1, max_lag + 1):
                th = dynamic_threshold(lag)

                lc = corr_level(x, y, lag)
                if lc is not None and abs(lc) >= th:
                    level_good.append(lag)

                dc = corr_lagdiff(x, y, lag)
                if dc is not None and abs(dc) >= th:
                    diff_good.append(lag)

            # ---------------------------
            # Lag 개수 조건
            # ---------------------------
            if len(level_good) + len(diff_good) < min_valid_lags:
                continue

            # ---------------------------
            # BEST LAG을 찾기
            # ---------------------------
            best_score = 0
            best_lag = None
            best_type = None

            for lag in set(level_good + diff_good):
                lv = corr_level(x, y, lag)
                df = corr_lagdiff(x, y, lag)

                s_lv = abs(lv) if lv is not None else 0
                s_df = abs(df) if df is not None else 0

                # diff > level이면 diff 선택
                if s_df > s_lv:
                    score = s_df * np.sign(df)
                    tp = "lagdiff"
                else:
                    score = s_lv * np.sign(lv)
                    tp = "level"

                if abs(score) > abs(best_score):
                    best_score = score
                    best_lag = lag
                    best_type = tp

            if best_lag is None:
                continue

            # ---------------------------
            # ROLLING 검증
            # ---------------------------
            if best_type == "level":
                roll = rolling_level(x, y, best_lag)
            else:
                roll = rolling_diff(x, y, best_lag)

            if roll is None:
                continue

            if roll["pear_ratio"] < min_pear_stable:
                continue
            if roll["agree_ratio"] < min_spear_stable:
                continue

            # ---------------------------
            # 최종 저장
            # ---------------------------
            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "num_level_lags": len(level_good),
                "num_diff_lags": len(diff_good)
            })

    return pd.DataFrame(results)


###############################################################################
# RUN
###############################################################################
pairs = find_comovement_pairs(pivot)
print("검출된 공행성 쌍:", len(pairs))
pairs.head()


100%|██████████| 100/100 [03:43<00:00,  2.23s/it]

검출된 공행성 쌍: 3140


,leading_item_id,following_item_id,best_lag,score,type,num_level_lags,num_diff_lags
0,AANGBULD,BJALXPFS,2,-0.526894,lagdiff,0,2
1,AANGBULD,BSRMSVTC,4,-0.417119,level,1,1
2,AANGBULD,EVBVXETX,6,0.502014,level,4,1
3,AANGBULD,FITUEHWN,4,0.591073,lagdiff,1,3
4,AANGBULD,FQCLOEXA,8,-0.435308,lagdiff,1,1


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm

EPS = 1e-6   # 로그 안정화용

################################################################################
# 1. Threshold
################################################################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


################################################################################
# 2. Level correlation (log 안정적)
################################################################################
def safe_corr_lag_level(x, y, lag, min_valid=9):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    xv = lx[:-lag]
    yv = ly[lag:]

    if len(xv) < min_valid:
        return 0.0
    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation
    if spear is None or np.isnan(spear):
        return 0.0

    if pear * spear <= 0:
        return 0.0

    return pear


################################################################################
# 3. Lag-diff correlation  (핵심!!  변화량(t)-변화량(t-lag))
################################################################################
def safe_corr_lag_diff(x, y, lag, min_valid=9):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    # lag-length 변화량
    dx = lx[lag:] - lx[:-lag]      # A[t] - A[t-lag]
    dy = ly[lag:] - ly[:-lag]      # B[t] - B[t-lag]

    # shift: A influences B after lag
    if len(dx) <= 2 * lag:
        return 0.0

    dx2 = dx[:-lag]
    dy2 = dy[lag:]

    if len(dx2) < min_valid:
        return 0.0

    mask = ~((dx2 == 0) & (dy2 == 0))
    dxv = dx2[mask]
    dyv = dy2[mask]

    if len(dxv) < min_valid:
        return 0.0
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    pear = float(np.corrcoef(dxv, dyv)[0, 1])
    spear = spearmanr(dxv, dyv).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    if pear * spear <= 0:
        return 0.0

    return pear


################################################################################
# 4. lag-diff sign ratio (변화 방향 동의율)
################################################################################
def lagdiff_sign_ratio(x, y, lag, min_valid=6):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = lx[lag:] - lx[:-lag]
    dy = ly[lag:] - ly[:-lag]

    if len(dx) <= 2 * lag:
        return 0.0

    dx2 = dx[:-lag]
    dy2 = dy[lag:]

    mask = ~((dx2 == 0) & (dy2 == 0))
    dxv = dx2[mask]
    dyv = dy2[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))


################################################################################
# 5-1 Rolling for Level
################################################################################
def rolling_level_metrics(x, y, lag, window=12, min_windows=5):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    xv = lx[:-lag]
    yv = ly[lag:]
    T = len(xv)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = xv[start:start+window]
        ys = yv[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0, 1]
        spear = spearmanr(xs, ys).correlation
        if np.isnan(pear) or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pa = np.array(pear_list)
    main_sign = np.sign(np.mean(pa))

    return {
        "pear_sign_ratio": np.mean(np.sign(pa) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


################################################################################
# 5-2 Rolling for Lag-Diff
################################################################################
def rolling_lagdiff_metrics(x, y, lag, window=12, min_windows=5):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = lx[lag:] - lx[:-lag]
    dy = ly[lag:] - ly[:-lag]

    dx2 = dx[:-lag]
    dy2 = dy[lag:]
    T = len(dx2)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = dx2[start:start+window]
        ys = dy2[start:start+window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5 or np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0, 1]
        spear = spearmanr(xs2, ys2).correlation
        if np.isnan(pear) or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pa = np.array(pear_list)
    main_sign = np.sign(np.mean(pa))

    return {
        "pear_sign_ratio": np.mean(np.sign(pa) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


################################################################################
# 6. MASTER SEARCH (Level + LagDiff 둘 모두)
################################################################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_pear_consistency=0.60,
    min_spear_agree=0.60,
    diff_sign_threshold=0.60
):

    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if leader == follower:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            for lag in range(1, max_lag + 1):

                th = dynamic_threshold(lag)

                ############################################################################
                # Level correlation
                ############################################################################
                lv_corr = safe_corr_lag_level(x, y, lag)
                s_lv = abs(lv_corr) if abs(lv_corr) >= th else 0

                ############################################################################
                # Lag-Diff correlation
                ############################################################################
                df_corr = safe_corr_lag_diff(x, y, lag)
                df_sign = lagdiff_sign_ratio(x, y, lag)
                s_df = abs(df_corr) if (abs(df_corr) >= th and df_sign >= diff_sign_threshold) else 0

                # Choose winner
                if s_df > s_lv:
                    score = s_df * np.sign(df_corr)
                    tp = "lagdiff"
                else:
                    score = s_lv * np.sign(lv_corr)
                    tp = "level"

                if abs(score) > abs(best_score):
                    best_score = score
                    best_lag = lag
                    best_type = tp

            if best_lag is None:
                continue

            ############################################################################
            # Rolling verification
            ############################################################################
            if best_type == "level":
                roll = rolling_level_metrics(x, y, best_lag)
            else:
                roll = rolling_lagdiff_metrics(x, y, best_lag)

            if roll is None:
                continue

            if roll["pear_sign_ratio"] < min_pear_consistency:
                continue
            if roll["spearman_agree_ratio"] < min_spear_agree:
                continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_pear_ratio": roll["pear_sign_ratio"],
                "roll_spear_ratio": roll["spearman_agree_ratio"]
            })

    return pd.DataFrame(results)


################################################################################
# RUN
################################################################################
pairs = find_comovement_pairs(pivot)
print("검출된 공행성 쌍:", len(pairs))
pairs.head()


In [ ]:
# 연속된 lag에서 최소 2개는 threshold 보다 큰 조건
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag threshold
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#########################################
# 2. Level corr (0 포함)
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):
    if len(x) <= lag:
        return 0.0

    xv = x[:-lag]
    yv = y[lag:]

    if len(xv) < min_valid:
        return 0.0
    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0
    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. Rolling LEVEL (0 포함)
#########################################
def rolling_level_metrics(x, y, lag, window=12, min_windows=5):

    xv = x[:-lag]
    yv = y[lag:]
    T = len(xv)

    if T <= window:
        return None

    pear_list = []
    agree_list = []  # Pearson × Spearman > 0 여부

    for start in range(0, T - window + 1):
        xs = xv[start:start+window]
        ys = yv[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0,1]
        spear = spearmanr(xs, ys).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pear_arr = np.array(pear_list)

    main_sign = np.sign(np.mean(pear_arr))

    return {
        "pear_sign_ratio": np.mean(np.sign(pear_arr) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


#########################################
# 4. 공행성 탐색 (+ lag≥2 필터)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_pear_consistency=0.60,
    min_spear_agree=0.60
):

    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if leader == follower:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            #########################################
            # (A) 모든 lag에서 corr 계산 후
            #     threshold 넘는 lag가 최소 2개 필요한 필터
            #########################################
            corr_dict = {}
            valid_lags = []

            for lag in range(1, max_lag + 1):
                corr = safe_corr_lag(x, y, lag)
                corr_dict[lag] = corr

                th = dynamic_threshold(lag)
                if abs(corr) >= th:
                    valid_lags.append(lag)

            # ----------- 핵심 필터 -----------
            if len(valid_lags) < 2:
                continue  # ← lag 한 개만 넘으면 버림

            #########################################
            # (B) valid_lags 중 가장 강한 lag 선택
            #########################################
            best_lag = max(valid_lags, key=lambda L: abs(corr_dict[L]))
            best_corr = corr_dict[best_lag]

            #########################################
            # (C) Rolling 검증
            #########################################
            roll = rolling_level_metrics(x, y, best_lag)
            if roll is None:
                continue

            if roll["pear_sign_ratio"] < min_pear_consistency:
                continue
            if roll["spearman_agree_ratio"] < min_spear_agree:
                continue

            #########################################
            # (D) 채택
            #########################################
            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_corr,
                "type": "level",
                "roll_pear_ratio": roll["pear_sign_ratio"],
                "roll_spear_ratio": roll["spearman_agree_ratio"]
            })

    return pd.DataFrame(results)



#########################################
# 실행
#########################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 :", len(pairs))
pairs.head()


100%|██████████| 100/100 [01:44<00:00,  1.05s/it]

탐색된 공행성쌍 : 1163


,leading_item_id,following_item_id,best_lag,score,type,roll_pear_ratio,roll_spear_ratio
0,AANGBULD,APQGTRMF,8,-0.480115,level,0.958333,0.833333
1,AANGBULD,DEWLVASR,6,0.640221,level,0.807692,0.884615
2,AANGBULD,ELQGMQWE,10,0.507917,level,0.863636,0.818182
3,AANGBULD,EVBVXETX,6,0.436623,level,1.000000,0.961538
4,AANGBULD,FTSVTTSR,3,0.531400,level,0.620690,0.689655


In [ ]:
# 부호를 고려하는 각 롤링창에서도 pearson x spearman >0 인 비율 까지 고려 이거는 해볼지 말지 나중에 생각하기
from scipy.stats import spearmanr

#######################################
# 1. lag 구간별 threshold
#######################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.35


#######################################
# 2. Main Pearson + Spearman sign check
#######################################
def safe_corr_lag(x, y, lag, max_zero_ratio=0.5, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    L = len(x_shifted)

    if L < min_valid:
        return 0.0

    if np.std(x_shifted) == 0 or np.std(y_shifted) == 0:
        return 0.0

    zero_ratio = np.mean((x_shifted == 0) & (y_shifted == 0))
    if zero_ratio > max_zero_ratio:
        return 0.0

    pear = float(np.corrcoef(x_shifted, y_shifted)[0, 1])
    spear = spearmanr(x_shifted, y_shifted).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    # 💥 메인 부호 불일치 → 제거
    if pear * spear <= 0:
        return 0.0

    return pear


#######################################
# 3. Rolling 안정성 + Soft sign check
#######################################
def rolling_corr_metrics(x, y, lag, window=12, min_windows=5):
    """
    rolling windows:
      - pearson corr
      - pearson*spear_sign_ratio
    """
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)

    if T <= window:
        return None

    roll_corrs = []
    roll_sign_agree = []  # pearson*spear > 0 여부 저장

    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0, 1]
        spear = spearmanr(xs, ys).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        roll_corrs.append(pear)
        roll_sign_agree.append(pear * spear > 0)   # True/False 저장

    if len(roll_corrs) < min_windows:
        return None

    roll_corrs = np.array(roll_corrs)
    roll_sign_agree = np.array(roll_sign_agree)

    return {
        "corrs": roll_corrs,
        "sign_agree_ratio": roll_sign_agree.mean(),
        "std": roll_corrs.std(),
        "mean_abs": np.abs(roll_corrs).mean()
    }


#######################################
# 4. 공행성 탐색
#######################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,     # rolling pearson sign consistency
    min_sign_agree=0.60       # 🔥 rolling Pearson×Spearman의 soft sign agreement
):

    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            ###################################
            # best lag 찾기
            ###################################
            best_lag = None
            best_corr = 0.0

            for lag in range(1, max_lag + 1):
                if lag >= n_months:
                    continue

                corr = safe_corr_lag(x, y, lag)
                th = dynamic_threshold(lag)

                if abs(corr) >= th:
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag is None:
                continue

            ###################################
            # rolling 안정성 + soft sign 검사
            ###################################
            metrics = rolling_corr_metrics(x, y, best_lag)
            if metrics is None:
                continue

            sign_main = np.sign(best_corr)

            # pearson 기준 동일 부호 비율
            same_sign_ratio = np.mean(np.sign(metrics["corrs"]) == sign_main)

            if same_sign_ratio < min_consistency:
                continue

            # pearson*spear 부호 일치 비율 → soft check
            if metrics["sign_agree_ratio"] < min_sign_agree:
                continue

            ###################################
            # 최종 채택
            ###################################
            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "max_corr": best_corr,
                "roll_pearson_sign_ratio": same_sign_ratio,
                "roll_spearman_sign_ratio": metrics["sign_agree_ratio"],
                "roll_std": metrics["std"],
                "roll_mean_abs": metrics["mean_abs"]
            })

    return pd.DataFrame(results)


#######################################
# 실행
#######################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()


100it [02:09,  1.30s/it]

탐색된 공행성쌍 수: 2926


,leading_item_id,following_item_id,best_lag,max_corr,roll_pearson_sign_ratio,roll_spearman_sign_ratio,roll_std,roll_mean_abs
0,AANGBULD,APQGTRMF,8,-0.480115,0.958333,0.833333,0.183385,0.250263
1,AANGBULD,BJALXPFS,12,0.397642,0.600000,0.950000,0.436179,0.406486
2,AANGBULD,BLANHGYY,11,0.588076,0.809524,0.761905,0.264457,0.316035
3,AANGBULD,DDEXPPXU,2,0.383169,0.900000,0.833333,0.271188,0.387041
4,AANGBULD,DEWLVASR,6,0.640221,0.807692,0.884615,0.340005,0.496775


In [ ]:
# 부호를 고려하는 각 롤링창에서도 pearson x spearman >0 인 비율 까지 고려 이거는 해볼지 말지 나중에 생각하기
from scipy.stats import spearmanr

#######################################
# 1. lag 구간별 threshold
#######################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    elif lag <= 9:
        return 0.375
    else :
       return 0.4


#######################################
# 2. Main Pearson + Spearman sign check
#######################################
def safe_corr_lag(x, y, lag, max_zero_ratio=0.5, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    L = len(x_shifted)

    if L < min_valid:
        return 0.0

    if np.std(x_shifted) == 0 or np.std(y_shifted) == 0:
        return 0.0

    zero_ratio = np.mean((x_shifted == 0) & (y_shifted == 0))
    if zero_ratio > max_zero_ratio:
        return 0.0

    pear = float(np.corrcoef(x_shifted, y_shifted)[0, 1])
    spear = spearmanr(x_shifted, y_shifted).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    # 💥 메인 부호 불일치 → 제거
    if pear * spear <= 0:
        return 0.0

    return pear


#######################################
# 3. Rolling 안정성 + Soft sign check
#######################################
def rolling_corr_metrics(x, y, lag, window=12, min_windows=5):
    """
    rolling windows:
      - pearson corr
      - pearson*spear_sign_ratio
    """
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)

    if T <= window:
        return None

    roll_corrs = []
    roll_sign_agree = []  # pearson*spear > 0 여부 저장

    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0, 1]
        spear = spearmanr(xs, ys).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        roll_corrs.append(pear)
        roll_sign_agree.append(pear * spear > 0)   # True/False 저장

    if len(roll_corrs) < min_windows:
        return None

    roll_corrs = np.array(roll_corrs)
    roll_sign_agree = np.array(roll_sign_agree)

    return {
        "corrs": roll_corrs,
        "sign_agree_ratio": roll_sign_agree.mean(),
        "std": roll_corrs.std(),
        "mean_abs": np.abs(roll_corrs).mean()
    }


#######################################
# 4. 공행성 탐색
#######################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,     # rolling pearson sign consistency
    min_sign_agree=0.60       # 🔥 rolling Pearson×Spearman의 soft sign agreement
):

    items = pivot.index.to_list()
    n_months = len(pivot.columns)
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            ###################################
            # best lag 찾기
            ###################################
            best_lag = None
            best_corr = 0.0

            for lag in range(1, max_lag + 1):
                if lag >= n_months:
                    continue

                corr = safe_corr_lag(x, y, lag)
                th = dynamic_threshold(lag)

                if abs(corr) >= th:
                    if abs(corr) > abs(best_corr):
                        best_corr = corr
                        best_lag = lag

            if best_lag is None:
                continue

            ###################################
            # rolling 안정성 + soft sign 검사
            ###################################
            metrics = rolling_corr_metrics(x, y, best_lag)
            if metrics is None:
                continue

            sign_main = np.sign(best_corr)

            # pearson 기준 동일 부호 비율
            same_sign_ratio = np.mean(np.sign(metrics["corrs"]) == sign_main)

            if same_sign_ratio < min_consistency:
                continue

            # pearson*spear 부호 일치 비율 → soft check
            if metrics["sign_agree_ratio"] < min_sign_agree:
                continue

            ###################################
            # 최종 채택
            ###################################
            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "max_corr": best_corr,
                "roll_pearson_sign_ratio": same_sign_ratio,
                "roll_spearman_sign_ratio": metrics["sign_agree_ratio"],
                "roll_std": metrics["std"],
                "roll_mean_abs": metrics["mean_abs"]
            })

    return pd.DataFrame(results)


#######################################
# 실행
#######################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()


100it [02:01,  1.21s/it]

탐색된 공행성쌍 수: 2539


,leading_item_id,following_item_id,best_lag,max_corr,roll_pearson_sign_ratio,roll_spearman_sign_ratio,roll_std,roll_mean_abs
0,AANGBULD,APQGTRMF,8,-0.480115,0.958333,0.833333,0.183385,0.250263
1,AANGBULD,BLANHGYY,11,0.588076,0.809524,0.761905,0.264457,0.316035
2,AANGBULD,DDEXPPXU,2,0.383169,0.900000,0.833333,0.271188,0.387041
3,AANGBULD,DEWLVASR,6,0.640221,0.807692,0.884615,0.340005,0.496775
4,AANGBULD,DNMPSKTB,4,-0.410635,0.964286,0.857143,0.256069,0.378640


In [ ]:
def build_training_data(pivot, pairs):
    """
    공행성쌍 + 시계열을 이용해 (X, y) 학습 데이터를 만드는 함수
    input X:
      - b_t, b_t_1, a_t_lag, max_corr, best_lag
    target y:
      - b_t_plus_1
    """
    months = pivot.columns.to_list()
    n_months = len(months)

    rows = []

    for row in pairs.itertuples(index=False):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # t+1이 존재하고, t-lag >= 0인 구간만 학습에 사용
        for t in range(max(lag, 1), n_months - 1):
            b_t = b_series[t]
            b_t_1 = b_series[t - 1]
            a_t_lag = a_series[t - lag]
            b_t_plus_1 = b_series[t + 1]

            rows.append({
                "b_t": b_t,
                "b_t_1": b_t_1,
                "a_t_lag": a_t_lag,
                "target": b_t_plus_1,
            })

    df_train = pd.DataFrame(rows)
    return df_train

df_train_model = build_training_data(pivot, pairs)
print('생성된 학습 데이터의 shape :', df_train_model.shape)
df_train_model.head()

생성된 학습 데이터의 shape : (107788, 4)


,b_t,b_t_1,a_t_lag,target
0,331472.0,218947.0,14276.0,17480.0
1,17480.0,331472.0,52347.0,234330.0
2,234330.0,17480.0,53549.0,51692.0
3,51692.0,234330.0,0.0,164195.0
4,164195.0,51692.0,26997.0,59646.0


In [ ]:
# 회귀모델 학습
feature_cols = ['b_t', 'b_t_1', 'a_t_lag']

train_X = df_train_model[feature_cols].values
train_y = df_train_model["target"].values

reg = LinearRegression()
reg.fit(train_X, train_y)

LinearRegression()

In [ ]:
def predict(pivot, pairs, reg):
    months = pivot.columns.to_list()
    n_months = len(months)

    # 가장 마지막 두 달 index (2025-7, 2025-6)
    t_last = n_months - 1
    t_prev = n_months - 2

    preds = []

    for row in tqdm(pairs.itertuples(index=False)):
        leader = row.leading_item_id
        follower = row.following_item_id
        lag = int(row.best_lag)

        if leader not in pivot.index or follower not in pivot.index:
            continue

        a_series = pivot.loc[leader].values.astype(float)
        b_series = pivot.loc[follower].values.astype(float)

        # t_last - lag 가 0 이상인 경우만 예측
        if t_last - lag < 0:
            continue

        b_t = b_series[t_last]
        b_t_1 = b_series[t_prev]
        a_t_lag = a_series[t_last - lag]

        X_test = np.array([[b_t, b_t_1, a_t_lag]])
        y_pred = reg.predict(X_test)[0]

        # (후처리 1) 음수 예측 → 0으로 변환
        # (후처리 2) 소수점 → 정수 변환 (무역량은 정수 단위)
        y_pred = max(0.0, float(y_pred))
        y_pred = int(round(y_pred))

        preds.append({
            "leading_item_id": leader,
            "following_item_id": follower,
            "value": y_pred,
        })

    df_pred = pd.DataFrame(preds)
    return df_pred

In [ ]:
submission = predict(pivot, pairs, reg)
submission.head()

3140it [00:00, 4217.30it/s]


,leading_item_id,following_item_id,value
0,AANGBULD,BJALXPFS,301851
1,AANGBULD,BSRMSVTC,498193
2,AANGBULD,EVBVXETX,4971975
3,AANGBULD,FITUEHWN,212691
4,AANGBULD,FQCLOEXA,2172358


In [ ]:
submission.to_csv('./baseline_submit_lld2.csv', index=False)

In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag 구간별 threshold (level corr)
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.4


#########################################
# 2. level Pearson + Spearman sign check
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    # 0-0 제거
    mask = ~((x_shifted == 0) & (y_shifted == 0))
    xv = x_shifted[mask]
    yv = y_shifted[mask]

    if len(xv) < min_valid:
        return 0.0

    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. diff 기반 correlation
#########################################
def safe_corr_lag_diff(x, y, lag, min_valid=9):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    # 0-0 diff 제외
    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0

    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0,1])


#########################################
# 4. diff 부호 일치율
#########################################
def diff_sign_ratio(x, y, lag, min_valid=6):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))


#########################################
# 5. Rolling level corr 검사 (기존 유지)
#########################################
def rolling_sign_consistency(x, y, lag, window=12, min_windows=5):
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)

    if T <= window:
        return None

    roll_corrs = []

    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        # 0-0 제거
        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        c = np.corrcoef(xs2, ys2)[0, 1]
        if np.isnan(c):
            continue

        roll_corrs.append(c)

    if len(roll_corrs) < min_windows:
        return None

    return np.array(roll_corrs)


#########################################
# 6. 공행성 탐색 (level + diff 병합)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,
    diff_corr_threshold=0.35,
    diff_sign_threshold=0.60
):
    items = pivot.index.to_list()
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero: continue
        if np.count_nonzero(x) < min_nonzero: continue

        for follower in items:
            if follower == leader: continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero: continue
            if np.count_nonzero(y) < min_nonzero: continue

            best_lag = None
            best_score = 0

            for lag in range(1, max_lag + 1):

                # ---------- level corr ----------
                level_corr = safe_corr_lag(x, y, lag)
                th = dynamic_threshold(lag)

                # ---------- diff corr ----------
                d_corr = safe_corr_lag_diff(x, y, lag)

                # ---------- diff sign ----------
                d_sign = diff_sign_ratio(x, y, lag)

                # ---- 둘 중 강한 쪽을 채택 ----
                corr_score = 0

                if abs(level_corr) >= th:
                    corr_score = abs(level_corr)

                if abs(d_corr) >= diff_corr_threshold and d_sign >= diff_sign_threshold:
                    corr_score = max(corr_score, abs(d_corr))

                if corr_score > abs(best_score):
                    best_score = corr_score
                    best_lag = lag

            if best_lag is None:
                continue

            # Rolling level 기반 최종 필터
            roll_corrs = rolling_sign_consistency(x, y, best_lag)
            if roll_corrs is None: continue

            sign_main = np.sign(best_score)
            same_sign_ratio = np.mean(np.sign(roll_corrs) == sign_main)
            if same_sign_ratio < min_consistency: continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "roll_sign_ratio": same_sign_ratio
            })

    return pd.DataFrame(results)
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()

100it [02:06,  1.27s/it]

탐색된 공행성쌍 수: 2525


,leading_item_id,following_item_id,best_lag,score,roll_sign_ratio
0,AANGBULD,BEZYMBBT,10,0.483240,0.636364
1,AANGBULD,BJALXPFS,6,0.524327,0.846154
2,AANGBULD,BLANHGYY,11,0.572804,0.666667
3,AANGBULD,DDEXPPXU,2,0.383169,0.900000
4,AANGBULD,DEWLVASR,6,0.637443,0.807692


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag 구간별 threshold (level corr)
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.4


#########################################
# 2. level Pearson + Spearman sign check
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    # 0-0 제거
    mask = ~((x_shifted == 0) & (y_shifted == 0))
    xv = x_shifted[mask]
    yv = y_shifted[mask]

    if len(xv) < min_valid:
        return 0.0

    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. diff 기반 correlation
#########################################
def safe_corr_lag_diff(x, y, lag, min_valid=9):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    # 0-0 diff 제외
    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0

    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0,1])


#########################################
# 4. diff 부호 일치율
#########################################
def diff_sign_ratio(x, y, lag, min_valid=6):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))


#########################################
# 5. Rolling level corr 검사 (기존 유지)
#########################################
def rolling_sign_consistency(x, y, lag, window=12, min_windows=5):
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)

    if T <= window:
        return None

    roll_corrs = []

    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        # 0-0 제거
        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        c = np.corrcoef(xs2, ys2)[0, 1]
        if np.isnan(c):
            continue

        roll_corrs.append(c)

    if len(roll_corrs) < min_windows:
        return None

    return np.array(roll_corrs)


#########################################
# 6. 공행성 탐색 (level + diff 병합)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,
    diff_sign_threshold=0.60
):
    items = pivot.index.to_list()
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero: continue
        if np.count_nonzero(x) < min_nonzero: continue

        for follower in items:
            if follower == leader: continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero: continue
            if np.count_nonzero(y) < min_nonzero: continue

            best_lag = None
            best_score = 0

            for lag in range(1, max_lag + 1):

                # ---------- level corr ----------
                level_corr = safe_corr_lag(x, y, lag)
                th = dynamic_threshold(lag)

                # ---------- diff corr ----------
                d_corr = safe_corr_lag_diff(x, y, lag)

                # ---------- diff sign ----------
                d_sign = diff_sign_ratio(x, y, lag)

                # ---- 둘 중 강한 쪽을 채택 ----
                corr_score = 0

                if abs(level_corr) >= th:
                    corr_score = abs(level_corr)

                if abs(d_corr) >= th and d_sign >= diff_sign_threshold:
                    corr_score = max(corr_score, abs(d_corr))

                if corr_score > abs(best_score):
                    best_score = corr_score
                    best_lag = lag

            if best_lag is None:
                continue

            # Rolling level 기반 최종 필터
            roll_corrs = rolling_sign_consistency(x, y, best_lag)
            if roll_corrs is None: continue

            sign_main = np.sign(best_score)
            same_sign_ratio = np.mean(np.sign(roll_corrs) == sign_main)
            if same_sign_ratio < min_consistency: continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "roll_sign_ratio": same_sign_ratio
            })

    return pd.DataFrame(results)
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수:", len(pairs))
pairs.head()

100it [01:30,  1.11it/s]

탐색된 공행성쌍 수: 2324


,leading_item_id,following_item_id,best_lag,score,roll_sign_ratio
0,AANGBULD,BEZYMBBT,10,0.483240,0.636364
1,AANGBULD,BJALXPFS,6,0.524327,0.846154
2,AANGBULD,BLANHGYY,11,0.572804,0.666667
3,AANGBULD,DDEXPPXU,2,0.383169,0.900000
4,AANGBULD,DEWLVASR,6,0.637443,0.807692


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag 구간별 threshold (level corr)
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.4


#########################################
# 2. level Pearson + Spearman sign check
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    # 0-0 제거
    mask = ~((x_shifted == 0) & (y_shifted == 0))
    xv = x_shifted[mask]
    yv = y_shifted[mask]

    if len(xv) < min_valid:
        return 0.0

    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. diff 기반 correlation
#########################################
def safe_corr_lag_diff(x, y, lag, min_valid=9):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    # 0-0 diff 제외
    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0

    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0,1])


#########################################
# 4. diff 부호 일치율
#########################################
def diff_sign_ratio(x, y, lag, min_valid=6):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))

from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm

# ... (기존 1, 2, 3, 4번 함수들은 그대로 유지) ...

#########################################
# 5-1. Rolling LEVEL corr (기존)
#########################################
def rolling_sign_consistency(x, y, lag, window=12, min_windows=5):
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)

    if T <= window: return None

    roll_corrs = []
    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        # 0-0 제거
        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5: continue
        if np.std(xs2) == 0 or np.std(ys2) == 0: continue

        c = np.corrcoef(xs2, ys2)[0, 1]
        if np.isnan(c): continue
        roll_corrs.append(c)

    if len(roll_corrs) < min_windows: return None
    return np.array(roll_corrs)

#########################################
# 5-2. Rolling DIFF corr (🔥 신규 추가)
#########################################
def rolling_diff_consistency(x, y, lag, window=12, min_windows=5):
    # 전체 diff를 먼저 구함
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag: return None

    dx_shifted = dx[:-lag]
    dy_shifted = dy[lag:]
    T = len(dx_shifted)

    if T <= window: return None

    roll_corrs = []
    for start in range(0, T - window + 1):
        dxs = dx_shifted[start:start+window]
        dys = dy_shifted[start:start+window]

        # 0-0 diff 제거
        mask = ~((dxs == 0) & (dys == 0))
        dxs2 = dxs[mask]
        dys2 = dys[mask]

        if len(dxs2) < 5: continue
        if np.std(dxs2) == 0 or np.std(dys2) == 0: continue

        c = np.corrcoef(dxs2, dys2)[0, 1]
        if np.isnan(c): continue
        roll_corrs.append(c)

    if len(roll_corrs) < min_windows: return None
    return np.array(roll_corrs)

#########################################
# 6. 공행성 탐색 (맞춤형 검증 적용)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,
    diff_sign_threshold=0.60
):
    items = pivot.index.to_list()
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)
        if np.sum(x == 0) >= max_zero: continue
        if np.count_nonzero(x) < min_nonzero: continue

        for follower in items:
            if follower == leader: continue
            y = pivot.loc[follower].values.astype(float)
            if np.sum(y == 0) >= max_zero: continue
            if np.count_nonzero(y) < min_nonzero: continue

            best_lag = None
            best_score = 0
            best_type = None # 'level' or 'diff' 기록

            for lag in range(1, max_lag + 1):
                th = dynamic_threshold(lag)

                # 1. Level Score
                level_corr = safe_corr_lag(x, y, lag)
                s_level = abs(level_corr) if abs(level_corr) >= th else 0

                # 2. Diff Score
                d_corr = safe_corr_lag_diff(x, y, lag)
                d_sign = diff_sign_ratio(x, y, lag)
                # Diff는 sign ratio도 통과해야 점수 인정
                s_diff = abs(d_corr) if (abs(d_corr) >= th and d_sign >= diff_sign_threshold) else 0

                # 승자 결정
                current_max = max(s_level, s_diff)

                if current_max > abs(best_score):
                    best_score = current_max
                    best_lag = lag
                    # 누가 이겼는지 기록 (동점이면 level 우선)
                    if s_diff > s_level:
                        best_type = 'diff'
                    else:
                        best_type = 'level'

            if best_lag is None: continue

            # -------------------------------------------
            # 🔥 맞춤형 Rolling 검증 (Adaptive Validation)
            # -------------------------------------------
            roll_corrs = None

            if best_type == 'level':
                # Level로 뽑혔으면 Level Rolling 검증
                roll_corrs = rolling_sign_consistency(x, y, best_lag)
            else:
                # Diff로 뽑혔으면 Diff Rolling 검증
                roll_corrs = rolling_diff_consistency(x, y, best_lag)

            if roll_corrs is None: continue

            # 부호 일관성 체크
            # (주의: best_score는 절대값이므로, 원래 corr의 부호를 가져와야 함)
            # 여기서는 간단히 roll_corrs의 부호가 한쪽으로 쏠려있는지(majority vote)만 봐도 됨
            # 혹은 best_score가 양수라고 가정하고(abs했으니) roll_corrs도 양수여야 한다고 봐도 무방
            # 정확히 하려면 위에서 corr 원본 부호를 저장해야 하지만,
            # 보통 강한 관계는 양의 상관관계이거나 음의 상관관계가 명확함.

            # 여기서는 "롤링 상관계수들의 평균 부호"와 "개별 롤링 상관계수 부호"가 일치하는 비율을 봅니다.
            avg_sign = np.sign(np.mean(roll_corrs))
            if avg_sign == 0: avg_sign = 1 # 예외처리

            same_sign_ratio = np.mean(np.sign(roll_corrs) == avg_sign)

            if same_sign_ratio < min_consistency: continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type, # 나중에 분석용으로 유용
                "roll_sign_ratio": same_sign_ratio
            })

    return pd.DataFrame(results)

# 실행
pairs = find_comovement_pairs(pivot)
print(f"탐색된 쌍: {len(pairs)}")
print(pairs['type'].value_counts()) # level vs diff 비율 확인

100it [01:36,  1.04it/s]

탐색된 쌍: 3115
type
level    2098
diff     1017
Name: count, dtype: int64


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag 구간별 threshold (level corr)
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#########################################
# 2. level Pearson + Spearman sign check
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    # 0-0 제거
    mask = ~((x_shifted == 0) & (y_shifted == 0))
    xv = x_shifted[mask]
    yv = y_shifted[mask]

    if len(xv) < min_valid:
        return 0.0
    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    # 부호 불일치 제거
    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. diff 기반 correlation
#########################################
def safe_corr_lag_diff(x, y, lag, min_valid=9):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    # 0-0 diff 제거
    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0, 1])


#########################################
# 4. diff 부호 일치율
#########################################
def diff_sign_ratio(x, y, lag, min_valid=6):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    # 0-0 제거
    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))



#########################################
# 5-1. Rolling LEVEL
#########################################
def rolling_sign_consistency(x, y, lag, window=12, min_windows=5):
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)

    if T <= window:
        return None

    roll_corrs = []

    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        c = np.corrcoef(xs2, ys2)[0, 1]
        if not np.isnan(c):
            roll_corrs.append(c)

    if len(roll_corrs) < min_windows:
        return None

    return np.array(roll_corrs)



#########################################
# 5-2. Rolling DIFF
#########################################
def rolling_diff_consistency(x, y, lag, window=12, min_windows=5):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return None

    dx_shifted = dx[:-lag]
    dy_shifted = dy[lag:]
    T = len(dx_shifted)

    if T <= window:
        return None

    roll_corrs = []

    for start in range(0, T - window + 1):
        dxs = dx_shifted[start:start+window]
        dys = dy_shifted[start:start+window]

        mask = ~((dxs == 0) & (dys == 0))
        dxs2 = dxs[mask]
        dys2 = dys[mask]

        if len(dxs2) < 5:
            continue
        if np.std(dxs2) == 0 or np.std(dys2) == 0:
            continue

        c = np.corrcoef(dxs2, dys2)[0, 1]
        if not np.isnan(c):
            roll_corrs.append(c)

    if len(roll_corrs) < min_windows:
        return None

    return np.array(roll_corrs)



#########################################
# 6. 공행성 탐색 (level + diff, diff는 +0.10 더 강해야 채택)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,
    diff_sign_threshold=0.60,
    diff_advantage=0.10
):
    items = pivot.index.to_list()
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)
        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            for lag in range(1, max_lag + 1):

                th = dynamic_threshold(lag)

                # level
                level_corr = safe_corr_lag(x, y, lag)
                s_level = abs(level_corr)

                # diff
                d_corr = safe_corr_lag_diff(x, y, lag)
                d_sign = diff_sign_ratio(x, y, lag)
                s_diff = abs(d_corr)

                chosen = False

                # ----------- diff 우선 체크 -----------
                if (s_diff >= th) and (d_sign >= diff_sign_threshold) and (s_diff >= s_level + diff_advantage):
                    curr_score = s_diff
                    curr_type = "diff"
                    chosen = True

                # ----------- diff 실패 → level 체크 -----------
                elif (s_level >= th):
                    curr_score = s_level
                    curr_type = "level"
                    chosen = True

                if not chosen:
                    continue

                # best 갱신
                if curr_score > abs(best_score):
                    best_score = curr_score
                    best_lag = lag
                    best_type = curr_type

            if best_lag is None:
                continue

            # Rolling 검증
            if best_type == "level":
                roll_corrs = rolling_sign_consistency(x, y, best_lag)
            else:
                roll_corrs = rolling_diff_consistency(x, y, best_lag)

            if roll_corrs is None:
                continue

            sign_main = np.sign(best_score)
            same_sign_ratio = np.mean(np.sign(roll_corrs) == sign_main)

            if same_sign_ratio < min_consistency:
                continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_sign_ratio": same_sign_ratio
            })

    return pd.DataFrame(results)



#########################################
# 실행
#########################################
pairs = find_comovement_pairs(pivot)

print("탐색된 공행성쌍 수:", len(pairs))
print(pairs['type'].value_counts())
pairs.head()


100it [01:29,  1.11it/s]

탐색된 공행성쌍 수: 2282
type
level    1522
diff      760
Name: count, dtype: int64


,leading_item_id,following_item_id,best_lag,score,type,roll_sign_ratio
0,AANGBULD,BEZYMBBT,10,0.483240,level,0.636364
1,AANGBULD,BJALXPFS,6,0.524327,diff,0.920000
2,AANGBULD,BLANHGYY,11,0.572804,level,0.666667
3,AANGBULD,DDEXPPXU,2,0.383169,level,0.900000
4,AANGBULD,DEWLVASR,6,0.637443,level,0.807692


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag 구간별 threshold (level corr)
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#########################################
# 2. level Pearson + Spearman sign check
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    # 0-0 제거
    mask = ~((x_shifted == 0) & (y_shifted == 0))
    xv = x_shifted[mask]
    yv = y_shifted[mask]

    if len(xv) < min_valid:
        return 0.0
    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    # 부호 불일치 제거
    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. diff 기반 correlation
#########################################
def safe_corr_lag_diff(x, y, lag, min_valid=9):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    # 0-0 diff 제거
    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0, 1])


#########################################
# 4. diff 부호 일치율
#########################################
def diff_sign_ratio(x, y, lag, min_valid=6):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    # 0-0 제거
    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))



#########################################
# 5-1. Rolling LEVEL
#########################################
def rolling_sign_consistency(x, y, lag, window=12, min_windows=5):
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)

    if T <= window:
        return None

    roll_corrs = []

    for start in range(0, T - window + 1):
        xs = x_shifted[start:start+window]
        ys = y_shifted[start:start+window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        c = np.corrcoef(xs2, ys2)[0, 1]
        if not np.isnan(c):
            roll_corrs.append(c)

    if len(roll_corrs) < min_windows:
        return None

    return np.array(roll_corrs)



#########################################
# 5-2. Rolling DIFF
#########################################
def rolling_diff_consistency(x, y, lag, window=12, min_windows=5):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return None

    dx_shifted = dx[:-lag]
    dy_shifted = dy[lag:]
    T = len(dx_shifted)

    if T <= window:
        return None

    roll_corrs = []

    for start in range(0, T - window + 1):
        dxs = dx_shifted[start:start+window]
        dys = dy_shifted[start:start+window]

        mask = ~((dxs == 0) & (dys == 0))
        dxs2 = dxs[mask]
        dys2 = dys[mask]

        if len(dxs2) < 5:
            continue
        if np.std(dxs2) == 0 or np.std(dys2) == 0:
            continue

        c = np.corrcoef(dxs2, dys2)[0, 1]
        if not np.isnan(c):
            roll_corrs.append(c)

    if len(roll_corrs) < min_windows:
        return None

    return np.array(roll_corrs)



#########################################
# 6. 공행성 탐색 (level + diff, diff는 +0.10 더 강해야 채택)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,
    diff_sign_threshold=0.60,
    diff_advantage=0.05
):
    items = pivot.index.to_list()
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)
        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            for lag in range(1, max_lag + 1):

                th = dynamic_threshold(lag)

                # level
                level_corr = safe_corr_lag(x, y, lag)
                s_level = abs(level_corr)

                # diff
                d_corr = safe_corr_lag_diff(x, y, lag)
                d_sign = diff_sign_ratio(x, y, lag)
                s_diff = abs(d_corr)

                chosen = False

                # ----------- diff 우선 체크 -----------
                if (s_diff >= th) and (d_sign >= diff_sign_threshold) and (s_diff >= s_level + diff_advantage):
                    curr_score = s_diff
                    curr_type = "diff"
                    chosen = True

                # ----------- diff 실패 → level 체크 -----------
                elif (s_level >= th):
                    curr_score = s_level
                    curr_type = "level"
                    chosen = True

                if not chosen:
                    continue

                # best 갱신
                if curr_score > abs(best_score):
                    best_score = curr_score
                    best_lag = lag
                    best_type = curr_type

            if best_lag is None:
                continue

            # Rolling 검증
            if best_type == "level":
                roll_corrs = rolling_sign_consistency(x, y, best_lag)
            else:
                roll_corrs = rolling_diff_consistency(x, y, best_lag)

            if roll_corrs is None:
                continue

            sign_main = np.sign(best_score)
            same_sign_ratio = np.mean(np.sign(roll_corrs) == sign_main)

            if same_sign_ratio < min_consistency:
                continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_sign_ratio": same_sign_ratio
            })

    return pd.DataFrame(results)



#########################################
# 실행
#########################################
pairs = find_comovement_pairs(pivot)

print("탐색된 공행성쌍 수:", len(pairs))
print(pairs['type'].value_counts())
pairs.head()


100it [01:28,  1.12it/s]

탐색된 공행성쌍 수: 2344
type
level    1443
diff      901
Name: count, dtype: int64


,leading_item_id,following_item_id,best_lag,score,type,roll_sign_ratio
0,AANGBULD,BEZYMBBT,10,0.483240,level,0.636364
1,AANGBULD,BJALXPFS,6,0.524327,diff,0.920000
2,AANGBULD,BLANHGYY,11,0.572804,level,0.666667
3,AANGBULD,DDEXPPXU,2,0.383169,level,0.900000
4,AANGBULD,DEWLVASR,6,0.637443,level,0.807692


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag 구간별 threshold (level corr)
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#########################################
# 2. level Pearson + Spearman sign check
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    mask = ~((x_shifted == 0) & (y_shifted == 0))
    xv = x_shifted[mask]
    yv = y_shifted[mask]

    if len(xv) < min_valid:
        return 0.0

    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    # 부호 불일치 제거
    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. diff 기반 correlation
#########################################
def safe_corr_lag_diff(x, y, lag, min_valid=9):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0, 1])


#########################################
# 4. diff 부호 일치율
#########################################
def diff_sign_ratio(x, y, lag, min_valid=6):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))


#########################################
# 5-1. Rolling LEVEL pearson × spearman
#########################################
def rolling_level_sign_agree(x, y, lag, window=12, min_windows=5):
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)
    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = x_shifted[start:start + window]
        ys = y_shifted[start:start + window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0, 1]
        spear = spearmanr(xs2, ys2).correlation

        if np.isnan(pear) or (spear is None) or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    return {
        "pear": np.array(pear_list),
        "agree_ratio": np.mean(agree_list)
    }


#########################################
# 5-2. Rolling DIFF pearson × spearman
#########################################
def rolling_diff_sign_agree(x, y, lag, window=12, min_windows=5):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return None

    dx_shifted = dx[:-lag]
    dy_shifted = dy[lag:]

    T = len(dx_shifted)
    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = dx_shifted[start:start + window]
        ys = dy_shifted[start:start + window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0, 1]
        spear = spearmanr(xs2, ys2).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    return {
        "pear": np.array(pear_list),
        "agree_ratio": np.mean(agree_list)
    }


#########################################
# 6. 공행성 탐색 (level + diff + rolling agree)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,
    diff_sign_threshold=0.60
):

    items = pivot.index.to_list()
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)
        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue
            y = pivot.loc[follower].values.astype(float)
            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            for lag in range(1, max_lag + 1):
                th = dynamic_threshold(lag)

                # Level score
                level_corr = safe_corr_lag(x, y, lag)
                s_level = abs(level_corr) if abs(level_corr) >= th else 0

                # Diff score
                d_corr = safe_corr_lag_diff(x, y, lag)
                d_sign = diff_sign_ratio(x, y, lag)
                s_diff = abs(d_corr) if (abs(d_corr) >= th and d_sign >= diff_sign_threshold) else 0

                # 결정 (diff는 noise 억제)
                if s_diff > s_level + 0.05:
                    winner_score = s_diff * np.sign(d_corr)
                    winner_type = "diff"
                else:
                    winner_score = s_level * np.sign(level_corr)
                    winner_type = "level"

                if abs(winner_score) > abs(best_score):
                    best_score = winner_score
                    best_lag = lag
                    best_type = winner_type

            if best_lag is None:
                continue

            # Rolling 검증
            if best_type == "level":
                roll = rolling_level_sign_agree(x, y, best_lag)
            else:
                roll = rolling_diff_sign_agree(x, y, best_lag)

            if roll is None:
                continue

            if roll["agree_ratio"] < min_consistency:
                continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_agree_ratio": roll["agree_ratio"]
            })

    return pd.DataFrame(results)


#########################################
# 실행
#########################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수 :", len(pairs))
pairs.head()


100it [02:14,  1.34s/it]

탐색된 공행성쌍 수 : 3355


,leading_item_id,following_item_id,best_lag,score,type,roll_agree_ratio
0,AANGBULD,APQGTRMF,8,-0.480115,level,0.833333
1,AANGBULD,BEZYMBBT,10,-0.483240,level,0.909091
2,AANGBULD,BJALXPFS,6,0.524327,diff,0.960000
3,AANGBULD,BLANHGYY,11,0.572804,level,0.714286
4,AANGBULD,DDEXPPXU,2,0.383169,level,0.833333


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag 구간별 threshold (level corr)
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#########################################
# 2. level Pearson + Spearman sign check
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):
    if len(x) <= lag:
        return 0.0

    x_shifted = x[:-lag]
    y_shifted = y[lag:]

    mask = ~((x_shifted == 0) & (y_shifted == 0))
    xv = x_shifted[mask]
    yv = y_shifted[mask]

    if len(xv) < min_valid:
        return 0.0

    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0

    # 부호 불일치 제거
    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. diff 기반 correlation
#########################################
def safe_corr_lag_diff(x, y, lag, min_valid=9):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0, 1])


#########################################
# 4. diff 부호 일치율
#########################################
def diff_sign_ratio(x, y, lag, min_valid=6):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_lag = dx[:-lag]
    dy_lag = dy[lag:]

    mask = ~((dx_lag == 0) & (dy_lag == 0))
    dxv = dx_lag[mask]
    dyv = dy_lag[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))


#########################################
# 5-1. Rolling LEVEL pearson × spearman
#########################################
def rolling_level_sign_agree(x, y, lag, window=12, min_windows=5):
    x_shifted = x[:-lag]
    y_shifted = y[lag:]
    T = len(x_shifted)
    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = x_shifted[start:start + window]
        ys = y_shifted[start:start + window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0, 1]
        spear = spearmanr(xs2, ys2).correlation

        if np.isnan(pear) or (spear is None) or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    return {
        "pear": np.array(pear_list),
        "agree_ratio": np.mean(agree_list)
    }


#########################################
# 5-2. Rolling DIFF pearson × spearman
#########################################
def rolling_diff_sign_agree(x, y, lag, window=12, min_windows=5):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return None

    dx_shifted = dx[:-lag]
    dy_shifted = dy[lag:]

    T = len(dx_shifted)
    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = dx_shifted[start:start + window]
        ys = dy_shifted[start:start + window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0, 1]
        spear = spearmanr(xs2, ys2).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    return {
        "pear": np.array(pear_list),
        "agree_ratio": np.mean(agree_list)
    }


#########################################
# 6. 공행성 탐색 (level + diff + rolling agree)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,
    diff_sign_threshold=0.60
):

    items = pivot.index.to_list()
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)
        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue
            y = pivot.loc[follower].values.astype(float)
            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            for lag in range(1, max_lag + 1):
                th = dynamic_threshold(lag)

                # Level score
                level_corr = safe_corr_lag(x, y, lag)
                s_level = abs(level_corr) if abs(level_corr) >= th else 0

                # Diff score
                d_corr = safe_corr_lag_diff(x, y, lag)
                d_sign = diff_sign_ratio(x, y, lag)
                s_diff = abs(d_corr) if (abs(d_corr) >= th and d_sign >= diff_sign_threshold) else 0

                # 결정 (diff는 noise 억제)
                if s_diff > s_level:
                    winner_score = s_diff * np.sign(d_corr)
                    winner_type = "diff"
                else:
                    winner_score = s_level * np.sign(level_corr)
                    winner_type = "level"

                if abs(winner_score) > abs(best_score):
                    best_score = winner_score
                    best_lag = lag
                    best_type = winner_type

            if best_lag is None:
                continue

            # Rolling 검증
            if best_type == "level":
                roll = rolling_level_sign_agree(x, y, best_lag)
            else:
                roll = rolling_diff_sign_agree(x, y, best_lag)

            if roll is None:
                continue

            if roll["agree_ratio"] < min_consistency:
                continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_agree_ratio": roll["agree_ratio"]
            })

    return pd.DataFrame(results)


#########################################
# 실행
#########################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수 :", len(pairs))
pairs.head()


100it [02:13,  1.33s/it]

탐색된 공행성쌍 수 : 3355


,leading_item_id,following_item_id,best_lag,score,type,roll_agree_ratio
0,AANGBULD,APQGTRMF,8,-0.480115,level,0.833333
1,AANGBULD,BEZYMBBT,10,-0.483240,level,0.909091
2,AANGBULD,BJALXPFS,6,0.524327,diff,0.960000
3,AANGBULD,BLANHGYY,11,0.572804,level,0.714286
4,AANGBULD,DDEXPPXU,2,0.383169,level,0.833333


In [ ]:
# tt
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag threshold
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#########################################
# 2. Level corr (0 제거 X)
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):

    if len(x) <= lag:
        return 0.0

    # 0 포함
    xv = x[:-lag]
    yv = y[lag:]

    if len(xv) < min_valid:
        return 0.0
    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0
    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. diff corr (0 제거 O)
#########################################
def safe_coll_lag_diff(x, y, lag, min_valid=9):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_shift = dx[:-lag]
    dy_shift = dy[lag:]

    mask = ~((dx_shift == 0) & (dy_shift == 0))
    dxv = dx_shift[mask]
    dyv = dy_shift[mask]

    if len(dxv) < min_valid:
        return 0.0
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0, 1])


#########################################
# 4. diff sign agree
#########################################
def diff_sign_ratio(x, y, lag, min_valid=6):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_shift = dx[:-lag]
    dy_shift = dy[lag:]

    mask = ~((dx_shift == 0) & (dy_shift == 0))
    dxv = dx_shift[mask]
    dyv = dy_shift[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))


#########################################
# 5-1 Rolling level (0 제거 X)
#########################################
def rolling_level_sign_agree(x, y, lag, window=12, min_windows=5):

    xv = x[:-lag]
    yv = y[lag:]
    T = len(xv)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):

        xs = xv[start:start+window]
        ys = yv[start:start+window]

        # 0 제거 안함
        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0,1]
        spear = spearmanr(xs, ys).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    return {
        "pear": np.array(pear_list),
        "agree_ratio": np.mean(agree_list)
    }


#########################################
# 5-2 Rolling diff (0 제거 O)
#########################################
def rolling_diff_sign_agree(x, y, lag, window=12, min_windows=5):

    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return None

    xv = dx[:-lag]
    yv = dy[lag:]
    T = len(xv)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):

        xs = xv[start:start+window]
        ys = yv[start:start+window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0,1]
        spear = spearmanr(xs2, ys2).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    return {
        "pear": np.array(pear_list),
        "agree_ratio": np.mean(agree_list)
    }


#########################################
# 6. Master search
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_consistency=0.60,
    diff_sign_threshold=0.60
):

    items = pivot.index.to_list()
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            # ---- lag loop ----
            for lag in range(1, max_lag+1):

                th = dynamic_threshold(lag)

                # level
                level_corr = safe_corr_lag(x, y, lag)
                s_level = abs(level_corr) if abs(level_corr) >= th else 0

                # diff
                d_corr = safe_coll_lag_diff(x, y, lag)
                d_sign = diff_sign_ratio(x, y, lag)
                s_diff = abs(d_corr) if (abs(d_corr) >= th and d_sign >= diff_sign_threshold) else 0

                # winner
                if s_diff > s_level:
                    score = s_diff * np.sign(d_corr)
                    tp = "diff"
                else:
                    score = s_level * np.sign(level_corr)
                    tp = "level"

                if abs(score) > abs(best_score):
                    best_score = score
                    best_lag = lag
                    best_type = tp

            if best_lag is None:
                continue

            # Rolling 검증
            if best_type == "level":
                roll = rolling_level_sign_agree(x, y, best_lag)
            else:
                roll = rolling_diff_sign_agree(x, y, best_lag)

            if roll is None:
                continue

            if roll["agree_ratio"] < min_consistency:
                continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_agree_ratio": roll["agree_ratio"]
            })

    return pd.DataFrame(results)



#########################################
# 실행
#########################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수 :", len(pairs))
pairs.head()


100it [02:12,  1.32s/it]

탐색된 공행성쌍 수 : 3360


,leading_item_id,following_item_id,best_lag,score,type,roll_agree_ratio
0,AANGBULD,APQGTRMF,8,-0.480115,level,0.833333
1,AANGBULD,BEZYMBBT,10,-0.483240,level,0.909091
2,AANGBULD,BJALXPFS,6,0.524327,diff,0.960000
3,AANGBULD,BLANHGYY,11,0.588076,level,0.761905
4,AANGBULD,DDEXPPXU,2,0.383169,level,0.833333


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag threshold
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#########################################
# 2. Level corr (0 포함)
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):

    if len(x) <= lag:
        return 0.0

    xv = x[:-lag]
    yv = y[lag:]

    if len(xv) < min_valid:
        return 0.0
    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0
    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. diff corr (0 제거)
#########################################
def safe_corr_lag_diff(x, y, lag, min_valid=9):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_shift = dx[:-lag]
    dy_shift = dy[lag:]

    mask = ~((dx_shift == 0) & (dy_shift == 0))
    dxv = dx_shift[mask]
    dyv = dy_shift[mask]

    if len(dxv) < min_valid:
        return 0.0
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0, 1])


#########################################
# 4. diff sign agree
#########################################
def diff_sign_ratio(x, y, lag, min_valid=6):
    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return 0.0

    dx_shift = dx[:-lag]
    dy_shift = dy[lag:]

    mask = ~((dx_shift == 0) & (dy_shift == 0))
    dxv = dx_shift[mask]
    dyv = dy_shift[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))


#########################################
# 5-1 Rolling LEVEL (0 포함)
#########################################
def rolling_level_metrics(x, y, lag, window=12, min_windows=5):

    xv = x[:-lag]
    yv = y[lag:]
    T = len(xv)

    if T <= window:
        return None

    pear_list = []
    agree_list = []
    pear_sign_list = []

    for start in range(0, T - window + 1):
        xs = xv[start:start+window]
        ys = yv[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0,1]
        spear = spearmanr(xs, ys).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)
        pear_sign_list.append(np.sign(pear))

    if len(pear_list) < min_windows:
        return None

    pear_arr = np.array(pear_list)
    main_sign = np.sign(np.mean(pear_arr))

    return {
        "pear": pear_arr,
        "pear_sign_ratio": np.mean(np.sign(pear_arr) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


#########################################
# 5-2 Rolling DIFF (0 제거)
#########################################
def rolling_diff_metrics(x, y, lag, window=12, min_windows=5):

    dx = np.diff(x)
    dy = np.diff(y)

    if len(dx) <= lag:
        return None

    xv = dx[:-lag]
    yv = dy[lag:]
    T = len(xv)

    if T <= window:
        return None

    pear_list = []
    agree_list = []
    pear_sign_list = []

    for start in range(0, T - window + 1):
        xs = xv[start:start+window]
        ys = yv[start:start+window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0,1]
        spear = spearmanr(xs2, ys2).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)
        pear_sign_list.append(np.sign(pear))

    if len(pear_list) < min_windows:
        return None

    pear_arr = np.array(pear_list)
    main_sign = np.sign(np.mean(pear_arr))

    return {
        "pear": pear_arr,
        "pear_sign_ratio": np.mean(np.sign(pear_arr) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


#########################################
# 6. Master search
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_pear_consistency=0.60,
    min_spear_agree=0.60,
    diff_sign_threshold=0.60
):

    items = pivot.index.to_list()
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            # lag 탐색
            for lag in range(1, max_lag+1):

                th = dynamic_threshold(lag)

                # level
                level_corr = safe_corr_lag(x, y, lag)
                s_level = abs(level_corr) if abs(level_corr) >= th else 0

                # diff
                d_corr = safe_corr_lag_diff(x, y, lag)
                d_sign = diff_sign_ratio(x, y, lag)
                s_diff = abs(d_corr) if (abs(d_corr) >= th and d_sign >= diff_sign_threshold) else 0

                # winner
                if s_diff > s_level:
                    score = s_diff * np.sign(d_corr)
                    tp = "diff"
                else:
                    score = s_level * np.sign(level_corr)
                    tp = "level"

                if abs(score) > abs(best_score):
                    best_score = score
                    best_lag = lag
                    best_type = tp

            if best_lag is None:
                continue

            # Rolling 검사
            if best_type == "level":
                roll = rolling_level_metrics(x, y, best_lag)
            else:
                roll = rolling_diff_metrics(x, y, best_lag)

            if roll is None:
                continue

            # Pearson sign consistency
            if roll["pear_sign_ratio"] < min_pear_consistency:
                continue

            # Pearson×Spearman soft agreement
            if roll["spearman_agree_ratio"] < min_spear_agree:
                continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_pear_ratio": roll["pear_sign_ratio"],
                "roll_spear_ratio": roll["spearman_agree_ratio"]
            })

    return pd.DataFrame(results)



#########################################
# 실행
#########################################
pairs = find_comovement_pairs(pivot)
print("탐색된 공행성쌍 수 :", len(pairs))
pairs.head()


100it [02:58,  1.78s/it]

탐색된 공행성쌍 수 : 3097


,leading_item_id,following_item_id,best_lag,score,type,roll_pear_ratio,roll_spear_ratio
0,AANGBULD,APQGTRMF,8,-0.480115,level,0.958333,0.833333
1,AANGBULD,BJALXPFS,6,0.524327,diff,0.920000,0.960000
2,AANGBULD,BLANHGYY,11,0.588076,level,0.809524,0.761905
3,AANGBULD,DDEXPPXU,2,0.383169,level,0.900000,0.833333
4,AANGBULD,DEWLVASR,6,0.640221,level,0.807692,0.884615


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm


#########################################
# 1. lag threshold
#########################################
def dynamic_threshold(lag):
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#########################################
# 2. Level corr (0 포함)
#########################################
def safe_corr_lag(x, y, lag, min_valid=9):
    if len(x) <= lag:
        return 0.0

    xv = x[:-lag]
    yv = y[lag:]

    if len(xv) < min_valid:
        return 0.0
    if np.std(xv) == 0 or np.std(yv) == 0:
        return 0.0

    pear = float(np.corrcoef(xv, yv)[0, 1])
    spear = spearmanr(xv, yv).correlation

    if spear is None or np.isnan(spear):
        return 0.0
    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. percent diff 기반 correlation (0 제거)
#########################################
def safe_corr_lag_pct(x, y, lag, min_valid=9):
    # percent diff
    px = np.diff(x) / (x[:-1] + 1e-6)
    py = np.diff(y) / (y[:-1] + 1e-6)

    if len(px) <= lag:
        return 0.0

    px_lag = px[:-lag]
    py_lag = py[lag:]

    # 둘 다 0인 변화 제거
    mask = ~((px_lag == 0) & (py_lag == 0))
    px2 = px_lag[mask]
    py2 = py_lag[mask]

    if len(px2) < min_valid:
        return 0.0
    if np.std(px2) == 0 or np.std(py2) == 0:
        return 0.0

    return float(np.corrcoef(px2, py2)[0, 1])


#########################################
# 4. percent diff sign ratio
#########################################
def pct_sign_ratio(x, y, lag, min_valid=6):
    px = np.diff(x) / (x[:-1] + 1e-6)
    py = np.diff(y) / (y[:-1] + 1e-6)

    if len(px) <= lag:
        return 0.0

    px_lag = px[:-lag]
    py_lag = py[lag:]

    mask = ~((px_lag == 0) & (py_lag == 0))
    px2 = px_lag[mask]
    py2 = py_lag[mask]

    if len(px2) < min_valid:
        return 0.0

    return np.mean(np.sign(px2) == np.sign(py2))


#########################################
# 5-1 Rolling LEVEL (0 포함)
#########################################
def rolling_level_metrics(x, y, lag, window=12, min_windows=5):

    xv = x[:-lag]
    yv = y[lag:]
    T = len(xv)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = xv[start:start+window]
        ys = yv[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0,1]
        spear = spearmanr(xs, ys).correlation
        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pear_arr = np.array(pear_list)
    main_sign = np.sign(np.mean(pear_arr))

    return {
        "pear": pear_arr,
        "pear_sign_ratio": np.mean(np.sign(pear_arr) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


#########################################
# 5-2 Rolling percent diff (0 제거)
#########################################
def rolling_pct_metrics(x, y, lag, window=12, min_windows=5):

    px = np.diff(x) / (x[:-1] + 1e-6)
    py = np.diff(y) / (y[:-1] + 1e-6)

    if len(px) <= lag:
        return None

    xv = px[:-lag]
    yv = py[lag:]
    T = len(xv)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = xv[start:start+window]
        ys = yv[start:start+window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]
        if len(xs2) < 5:
            continue
        if np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0,1]
        spear = spearmanr(xs2, ys2).correlation

        if np.isnan(pear) or spear is None or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pear_arr = np.array(pear_list)
    main_sign = np.sign(np.mean(pear_arr))

    return {
        "pear": pear_arr,
        "pear_sign_ratio": np.mean(np.sign(pear_arr) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


#########################################
# 6. Master search (% diff 반영)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_pear_consistency=0.60,
    min_spear_agree=0.60,
    diff_sign_threshold=0.60
):

    items = pivot.index.to_list()
    results = []

    for i, leader in tqdm(enumerate(items)):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if follower == leader:
                continue

            y = pivot.loc[follower].values.astype(float)
            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            #######################################################
            # lag 탐색: level vs percent diff 비교
            #######################################################
            for lag in range(1, max_lag+1):

                th = dynamic_threshold(lag)

                # --- level corr
                level_corr = safe_corr_lag(x, y, lag)
                s_level = abs(level_corr) if abs(level_corr) >= th else 0

                # --- percent diff corr
                pct_corr = safe_corr_lag_pct(x, y, lag)
                pct_sign = pct_sign_ratio(x, y, lag)
                s_pct = abs(pct_corr) if (abs(pct_corr) >= th and pct_sign >= diff_sign_threshold) else 0

                # winner
                if s_pct > s_level:
                    score = s_pct * np.sign(pct_corr)
                    tp = "pct"
                else:
                    score = s_level * np.sign(level_corr)
                    tp = "level"

                if abs(score) > abs(best_score):
                    best_score = score
                    best_lag = lag
                    best_type = tp

            if best_lag is None:
                continue

            #######################################################
            # Rolling 검사
            #######################################################
            if best_type == "level":
                roll = rolling_level_metrics(x, y, best_lag)
            else:
                roll = rolling_pct_metrics(x, y, best_lag)

            if roll is None:
                continue

            if roll["pear_sign_ratio"] < min_pear_consistency:
                continue
            if roll["spearman_agree_ratio"] < min_spear_agree:
                continue

            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_pear_ratio": roll["pear_sign_ratio"],
                "roll_spear_ratio": roll["spearman_agree_ratio"]
            })

    return pd.DataFrame(results)



#########################################
# 실행
#########################################
pairs = find_comovement_pairs(pivot)
print("검출된 공행성쌍:", len(pairs))
pairs.head()


100it [02:28,  1.49s/it]

검출된 공행성쌍: 2918


,leading_item_id,following_item_id,best_lag,score,type,roll_pear_ratio,roll_spear_ratio
0,AANGBULD,APQGTRMF,8,-0.480115,level,0.958333,0.833333
1,AANGBULD,BLANHGYY,11,0.588076,level,0.809524,0.761905
2,AANGBULD,DDEXPPXU,2,0.682830,pct,0.827586,0.965517
3,AANGBULD,DEWLVASR,6,0.640221,level,0.807692,0.884615
4,AANGBULD,DNMPSKTB,10,0.418043,pct,0.952381,0.952381


In [ ]:
print(pairs['type'].value_counts()) # level vs diff 비율 확인

type
level    2240
pct       678
Name: count, dtype: int64


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm

EPS = 1e-6


#########################################
# 1. lag threshold
#########################################
def dynamic_threshold(lag):
    # 형이 쓰던 구조 그대로
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#########################################
# 2. LOG LEVEL CORRELATION
#########################################
def safe_corr_lag_log(x, y, lag, min_valid=9):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    lx2 = lx[:-lag]
    ly2 = ly[lag:]

    if len(lx2) < min_valid:
        return 0.0
    if np.std(lx2) == 0 or np.std(ly2) == 0:
        return 0.0

    pear = float(np.corrcoef(lx2, ly2)[0, 1])
    spear = spearmanr(lx2, ly2).correlation

    if spear is None or np.isnan(spear):
        return 0.0
    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. LOG-DIFF CORRELATION
#########################################
def safe_corr_lag_logdiff(x, y, lag, min_valid=9):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = np.diff(lx)
    dy = np.diff(ly)

    if len(dx) <= lag:
        return 0.0

    dx2 = dx[:-lag]
    dy2 = dy[lag:]

    mask = ~((dx2 == 0) & (dy2 == 0))
    dxv = dx2[mask]
    dyv = dy2[mask]

    if len(dxv) < min_valid:
        return 0.0
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0, 1])


#########################################
# 4. LOG-DIFF SIGN RATIO
#########################################
def logdiff_sign_ratio(x, y, lag, min_valid=6):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = np.diff(lx)
    dy = np.diff(ly)

    if len(dx) <= lag:
        return 0.0

    dx2 = dx[:-lag]
    dy2 = dy[lag:]

    mask = ~((dx2 == 0) & (dy2 == 0))
    dxv = dx2[mask]
    dyv = dy2[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))


#########################################
# 5-1 Rolling LOG-LEVEL
#########################################
def rolling_log_level(x, y, lag, window=12, min_windows=5):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    lx2 = lx[:-lag]
    ly2 = ly[lag:]
    T = len(lx2)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = lx2[start:start+window]
        ys = ly2[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0, 1]
        spear = spearmanr(xs, ys).correlation

        if np.isnan(pear) or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pa = np.array(pear_list)
    main_sign = np.sign(np.mean(pa))

    return {
        "pear_sign_ratio": np.mean(np.sign(pa) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


#########################################
# 5-2 Rolling LOG-DIFF
#########################################
def rolling_log_diff(x, y, lag, window=12, min_windows=5):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = np.diff(lx)
    dy = np.diff(ly)

    dx2 = dx[:-lag]
    dy2 = dy[lag:]
    T = len(dx2)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = dx2[start:start+window]
        ys = dy2[start:start+window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5 or np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0, 1]
        spear = spearmanr(xs2, ys2).correlation

        if np.isnan(pear) or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pa = np.array(pear_list)
    main_sign = np.sign(np.mean(pa))

    return {
        "pear_sign_ratio": np.mean(np.sign(pa) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


#########################################
# 6. MASTER SEARCH (level + logdiff 둘 다)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_pear_consistency=0.60,
    min_spear_agree=0.60,
    diff_sign_threshold=0.60
):

    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if leader == follower:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            for lag in range(1, max_lag + 1):

                th = dynamic_threshold(lag)

                # --- 1) LOG LEVEL
                lv_corr = safe_corr_lag_log(x, y, lag)
                s_lv = abs(lv_corr) if abs(lv_corr) >= th else 0

                # --- 2) LOG DIFF
                df_corr = safe_corr_lag_logdiff(x, y, lag)
                df_sign = logdiff_sign_ratio(x, y, lag)
                s_df = abs(df_corr) if (abs(df_corr) >= th and df_sign >= diff_sign_threshold) else 0

                # WINNER
                if s_df > s_lv:
                    score = s_df * np.sign(df_corr)
                    tp = "logdiff"
                else:
                    score = s_lv * np.sign(lv_corr)
                    tp = "level"

                if abs(score) > abs(best_score):
                    best_score = score
                    best_lag = lag
                    best_type = tp

            if best_lag is None:
                continue

            # Rolling 검증
            if best_type == "level":
                roll = rolling_log_level(x, y, best_lag)
            else:
                roll = rolling_log_diff(x, y, best_lag)

            if roll is None:
                continue

            if roll["pear_sign_ratio"] < min_pear_consistency:
                continue
            if roll["spearman_agree_ratio"] < min_spear_agree:
                continue

            # FINAL
            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_pear_ratio": roll["pear_sign_ratio"],
                "roll_spear_ratio": roll["spearman_agree_ratio"]
            })

    return pd.DataFrame(results)



#########################################
# 실행
#########################################
pairs = find_comovement_pairs(pivot)
print("검출된 공행성 쌍:", len(pairs))
pairs.head()


100%|██████████| 100/100 [02:46<00:00,  1.67s/it]

검출된 공행성 쌍: 3161


,leading_item_id,following_item_id,best_lag,score,type,roll_pear_ratio,roll_spear_ratio
0,AANGBULD,AXULOHBQ,1,-0.459176,level,0.903226,0.806452
1,AANGBULD,BSRMSVTC,4,-0.417119,level,0.964286,0.892857
2,AANGBULD,DNMPSKTB,5,-0.350557,level,0.703704,0.851852
3,AANGBULD,ELQGMQWE,7,0.448132,level,1.000000,0.960000
4,AANGBULD,EVBVXETX,6,0.502014,level,1.000000,0.961538


In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from tqdm import tqdm

EPS = 1


#########################################
# 1. lag threshold
#########################################
def dynamic_threshold(lag):
    # 형이 쓰던 구조 그대로
    if lag <= 6:
        return 0.35
    else:
        return 0.40


#########################################
# 2. LOG LEVEL CORRELATION
#########################################
def safe_corr_lag_log(x, y, lag, min_valid=9):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    lx2 = lx[:-lag]
    ly2 = ly[lag:]

    if len(lx2) < min_valid:
        return 0.0
    if np.std(lx2) == 0 or np.std(ly2) == 0:
        return 0.0

    pear = float(np.corrcoef(lx2, ly2)[0, 1])
    spear = spearmanr(lx2, ly2).correlation

    if spear is None or np.isnan(spear):
        return 0.0
    if pear * spear <= 0:
        return 0.0

    return pear


#########################################
# 3. LOG-DIFF CORRELATION
#########################################
def safe_corr_lag_logdiff(x, y, lag, min_valid=9):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = np.diff(lx)
    dy = np.diff(ly)

    if len(dx) <= lag:
        return 0.0

    dx2 = dx[:-lag]
    dy2 = dy[lag:]

    mask = ~((dx2 == 0) & (dy2 == 0))
    dxv = dx2[mask]
    dyv = dy2[mask]

    if len(dxv) < min_valid:
        return 0.0
    if np.std(dxv) == 0 or np.std(dyv) == 0:
        return 0.0

    return float(np.corrcoef(dxv, dyv)[0, 1])


#########################################
# 4. LOG-DIFF SIGN RATIO
#########################################
def logdiff_sign_ratio(x, y, lag, min_valid=6):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = np.diff(lx)
    dy = np.diff(ly)

    if len(dx) <= lag:
        return 0.0

    dx2 = dx[:-lag]
    dy2 = dy[lag:]

    mask = ~((dx2 == 0) & (dy2 == 0))
    dxv = dx2[mask]
    dyv = dy2[mask]

    if len(dxv) < min_valid:
        return 0.0

    return np.mean(np.sign(dxv) == np.sign(dyv))


#########################################
# 5-1 Rolling LOG-LEVEL
#########################################
def rolling_log_level(x, y, lag, window=12, min_windows=5):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    lx2 = lx[:-lag]
    ly2 = ly[lag:]
    T = len(lx2)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = lx2[start:start+window]
        ys = ly2[start:start+window]

        if np.std(xs) == 0 or np.std(ys) == 0:
            continue

        pear = np.corrcoef(xs, ys)[0, 1]
        spear = spearmanr(xs, ys).correlation

        if np.isnan(pear) or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pa = np.array(pear_list)
    main_sign = np.sign(np.mean(pa))

    return {
        "pear_sign_ratio": np.mean(np.sign(pa) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


#########################################
# 5-2 Rolling LOG-DIFF
#########################################
def rolling_log_diff(x, y, lag, window=12, min_windows=5):

    lx = np.log(x + EPS)
    ly = np.log(y + EPS)

    dx = np.diff(lx)
    dy = np.diff(ly)

    dx2 = dx[:-lag]
    dy2 = dy[lag:]
    T = len(dx2)

    if T <= window:
        return None

    pear_list = []
    agree_list = []

    for start in range(0, T - window + 1):
        xs = dx2[start:start+window]
        ys = dy2[start:start+window]

        mask = ~((xs == 0) & (ys == 0))
        xs2 = xs[mask]
        ys2 = ys[mask]

        if len(xs2) < 5 or np.std(xs2) == 0 or np.std(ys2) == 0:
            continue

        pear = np.corrcoef(xs2, ys2)[0, 1]
        spear = spearmanr(xs2, ys2).correlation

        if np.isnan(pear) or np.isnan(spear):
            continue

        pear_list.append(pear)
        agree_list.append(pear * spear > 0)

    if len(pear_list) < min_windows:
        return None

    pa = np.array(pear_list)
    main_sign = np.sign(np.mean(pa))

    return {
        "pear_sign_ratio": np.mean(np.sign(pa) == main_sign),
        "spearman_agree_ratio": np.mean(agree_list)
    }


#########################################
# 6. MASTER SEARCH (level + logdiff 둘 다)
#########################################
def find_comovement_pairs(
    pivot,
    max_lag=12,
    min_nonzero=12,
    max_zero=15,
    min_pear_consistency=0.60,
    min_spear_agree=0.60,
    diff_sign_threshold=0.60
):

    items = pivot.index.to_list()
    results = []

    for leader in tqdm(items):
        x = pivot.loc[leader].values.astype(float)

        if np.sum(x == 0) >= max_zero:
            continue
        if np.count_nonzero(x) < min_nonzero:
            continue

        for follower in items:
            if leader == follower:
                continue

            y = pivot.loc[follower].values.astype(float)

            if np.sum(y == 0) >= max_zero:
                continue
            if np.count_nonzero(y) < min_nonzero:
                continue

            best_lag = None
            best_score = 0
            best_type = None

            for lag in range(1, max_lag + 1):

                th = dynamic_threshold(lag)

                # --- 1) LOG LEVEL
                lv_corr = safe_corr_lag_log(x, y, lag)
                s_lv = abs(lv_corr) if abs(lv_corr) >= th else 0

                # --- 2) LOG DIFF
                df_corr = safe_corr_lag_logdiff(x, y, lag)
                df_sign = logdiff_sign_ratio(x, y, lag)
                s_df = abs(df_corr) if (abs(df_corr) >= th and df_sign >= diff_sign_threshold) else 0

                # WINNER
                if s_df > s_lv:
                    score = s_df * np.sign(df_corr)
                    tp = "logdiff"
                else:
                    score = s_lv * np.sign(lv_corr)
                    tp = "level"

                if abs(score) > abs(best_score):
                    best_score = score
                    best_lag = lag
                    best_type = tp

            if best_lag is None:
                continue

            # Rolling 검증
            if best_type == "level":
                roll = rolling_log_level(x, y, best_lag)
            else:
                roll = rolling_log_diff(x, y, best_lag)

            if roll is None:
                continue

            if roll["pear_sign_ratio"] < min_pear_consistency:
                continue
            if roll["spearman_agree_ratio"] < min_spear_agree:
                continue

            # FINAL
            results.append({
                "leading_item_id": leader,
                "following_item_id": follower,
                "best_lag": best_lag,
                "score": best_score,
                "type": best_type,
                "roll_pear_ratio": roll["pear_sign_ratio"],
                "roll_spear_ratio": roll["spearman_agree_ratio"]
            })

    return pd.DataFrame(results)



#########################################
# 실행
#########################################
pairs = find_comovement_pairs(pivot)
print("검출된 공행성 쌍:", len(pairs))
pairs.head()


100%|██████████| 100/100 [02:36<00:00,  1.56s/it]

검출된 공행성 쌍: 3287


,leading_item_id,following_item_id,best_lag,score,type,roll_pear_ratio,roll_spear_ratio
0,AANGBULD,APQGTRMF,5,-0.444719,level,0.962963,1.000000
1,AANGBULD,AXULOHBQ,1,-0.494126,level,0.935484,0.838710
2,AANGBULD,BLANHGYY,2,-0.384603,level,1.000000,1.000000
3,AANGBULD,BSRMSVTC,4,-0.415788,level,0.964286,0.892857
4,AANGBULD,DDEXPPXU,2,0.383907,logdiff,0.862069,1.000000


In [ ]:
pairs1 = model1_result
pairs2 = model2_result
pairs3 = model3_result
pairs4 = model4_result
pairs5 = model5_result


In [ ]:
import pandas as pd
from collections import Counter
import numpy as np

##########################################################
# 🔥 1. 5개 모델의 결과를 리스트로 받기
##########################################################
models = [pairs1, pairs2, pairs3, pairs4, pairs5]

# 각 모델에 pair 문자열 부착
for df in models:
    df["pair"] = (
        df["leading_item_id"].astype(str) + "_" +
        df["following_item_id"].astype(str)
    )


##########################################################
# 🔥 2. 모든 모델에서 pair 등장 횟수 세기
##########################################################
pair_counts = Counter()
value_dict = {}   # pair → [value list]

for df in models:
    for _, row in df.iterrows():
        pair = row["pair"]
        pair_counts[pair] += 1

        if pair not in value_dict:
            value_dict[pair] = []
        value_dict[pair].append(row["max_corr"])   # value로 max_corr 사용


##########################################################
# 🔥 3. Vote 앙상블 기준 설정
##########################################################
VOTE_THRESHOLD = 2     # 최소 2개 모델에서 나온 pair만 채택 (추천)
# VOTE_THRESHOLD = 3   # 더 보수적으로 하고 싶으면 3개 이상으로 변경


##########################################################
# 🔥 4. 최종 앙상블 pair 추출
##########################################################
selected_pairs = [p for p, cnt in pair_counts.items() if cnt >= VOTE_THRESHOLD]

print("선택된 pair 수:", len(selected_pairs))

##########################################################
# 🔥 5. value 계산: median (또는 mean 사용 가능)
##########################################################
leading_ids = []
following_ids = []
values = []  # 최종 value (median 또는 mean)

for p in selected_pairs:
    lid, fid = p.split("_")
    leading_ids.append(int(lid))
    following_ids.append(int(fid))

    # value 결합 방식
    vals = value_dict[p]
    values.append(float(np.median(vals)))   # 중앙값 추천
    # values.append(float(np.mean(vals)))   # 평균 사용하려면 이걸로 변경


##########################################################
# 🔥 6. 최종 제출 dataframe 구성
##########################################################
ensemble_df = pd.DataFrame({
    "leading_item_id": leading_ids,
    "following_item_id": following_ids,
    "value": values
})

print(ensemble_df.head())
print("최종 pair 수:", len(ensemble_df))

##########################################################
# 🔥 7. 저장 (원하면)
##########################################################
# ensemble_df.to_csv("ensemble_vote_submission.csv", index=False)
